# 🛒 Supermarket AI — Detection, Tracking & Classification Dashboard

A college computer-vision prototype. It reads a supermarket video with **OpenCV**, finds products with your trained **YOLOv8** detector, follows them with **ByteTrack**, classifies each detected crop with your **grocery classifier**, and shows everything in a **Gradio dashboard** (overview, AI Studio, **Missing Products**, results table, CSV export).

**What you need** (upload to `/content`, or put in Google Drive and set the override paths in step 2):
- `best.pt` — your trained YOLOv8 detector (single class: `object`)
- `grocery_classifier_1143.zip` — your classifier checkpoint (ResNet18 state-dict, 25 classes)

**How to run:** `Runtime ▸ Run all` (a GPU is optional but makes analysis much faster). After the last cell starts, open the **public URL** it prints in a new browser tab. Upload a video in *AI Studio* and press **Analyze video**.

> **Honest limitations.** The uploaded files contain **no class-name mapping**, so classifier predictions appear as `class_0 … class_24` until you add a `class_names.json` (see step 6). The classifier's preprocessing is also not stored in the checkpoint; the standard ResNet recipe (224×224, ImageNet normalisation) is assumed. No accuracy or sales numbers are shown anywhere — every number on the dashboard comes from the video you analyse.

**New: Missing Products page.** Upload a reference shelf image and/or type a list of expected products, analyse the current shelf video, and see each expected product marked **Observed**, **Potentially Missing** or **Inconclusive**, with evidence crops and a CSV export. Every verdict comes from the real detector + classifier output for your video.

> **What “missing” can and cannot mean here.** The detector has one class (`object`) and the classifier has 25 classes, so the comparison is at **classifier-class level, not exact-product (SKU) level**. Without a `class_names.json`, typed product names cannot be matched; use the reference image (or type `class_N`). “Potentially Missing” is never proof of absence: occlusion, camera angle or a classifier mistake can all cause it, and thin evidence is reported as *Inconclusive*.


## 1. Install dependencies
Pinned to the versions this project was verified with. PyTorch/torchvision/numpy are left as Colab provides them. If Colab shows **“Restart session”**, click it and continue from step 2.

In [ ]:
# ---- Run this cell ONCE on a fresh Colab runtime ----
# Versions are pinned to the ones this project was verified against:
#   * ultralytics 8.4.138  -> the version that trained/saved your best.pt
#   * gradio 6.30.0        -> the version your Colab already resolved to
#   * lapx                 -> needed by ByteTrack (Ultralytics uses it for matching)
# PyTorch / torchvision / numpy are NOT touched: Colab already ships working ones.
%pip install -q "ultralytics==8.4.138" "gradio==6.30.0" lapx pandas

# OpenCV: keep exactly ONE variant installed (several variants share the `cv2` name
# and break each other). Headless = no GUI calls, which is what Colab needs.
%pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless
%pip install -q "opencv-python-headless==4.10.0.84"

print("\nInstall finished. If Colab shows a 'Restart session' banner, click it, then run from the next cell.")

## 2. Imports and version check

In [ ]:
import sys, os, glob, time, math, json, html, shutil, subprocess, traceback, inspect, zipfile, datetime, re, difflib

import cv2
import yaml
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
import gradio as gr

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INFER_DEVICE = 0 if torch.cuda.is_available() else "cpu"   # device argument for Ultralytics

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("Ultralytics :", ultralytics.__version__)
print("Gradio      :", gr.__version__)
print("OpenCV      :", cv2.__version__)
print("Device      :", DEVICE, "(GPU)" if DEVICE.type == "cuda" else "(CPU - analysis will be slower; use Runtime > Change runtime type > GPU if available)")

# Friendly warnings instead of mysterious crashes later
if not cv2.__version__.startswith("4.10"):
    print("\n[!] OpenCV is", cv2.__version__, "- expected 4.10.x. Run the install cell, then Runtime > Restart session.")
if gr.__version__ != "6.30.0":
    print("\n[!] Gradio is", gr.__version__, "- this notebook was tested with 6.30.0. Re-run the install cell and restart the session.")
if ultralytics.__version__ != "8.4.138":
    print("\n[!] Ultralytics is", ultralytics.__version__, "- best.pt was saved with 8.4.138. Re-run the install cell and restart the session.")

## 3. Configuration
All tunable settings live in one place.

In [ ]:
# ---- Project configuration (edit here, nowhere else) ----
DETECTOR_IMGSZ = 640            # YOLO input size (best.pt was trained with imgsz=640)
MAX_DET = 500                   # shelves can contain hundreds of products per frame
TRACKER_CONFIG = "bytetrack.yaml"   # ByteTrack config bundled with Ultralytics

# UI defaults
CONF_MIN, CONF_MAX, CONF_STEP, DEFAULT_CONF = 0.10, 0.90, 0.05, 0.40
FRAME_INTERVAL_CHOICES = [
    ("Every frame (interval 1)", 1),
    ("Every 2nd frame (interval 2)", 2),
    ("Every 3rd frame (interval 3)", 3),
    ("Every 5th frame (interval 5)", 5),
]
VALID_INTERVALS = [v for _, v in FRAME_INTERVAL_CHOICES]
DEFAULT_INTERVAL = 3
DEFAULT_MAX_PROCESSED = 100     # safety limit on analysed frames (keeps CPU runs short)

# Classifier preprocessing. ASSUMPTION (the checkpoint stores no transform):
# the standard torchvision ResNet recipe - RGB, 224x224, ImageNet mean/std.
CLS_INPUT_SIZE = 224
CLS_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 1, 3)
CLS_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 1, 3)
CLS_BATCH = 64

# ---- Missing Products page (all thresholds are editable here; the two classifier ones are also sliders) ----
MP_REF_MIN_CLS_CONF = 0.60            # reference image: a crop counts as a confident instance only above this classifier confidence
MP_DEFAULT_MIN_CLS_CONF = 0.60        # video: classifier confidence needed for a crop to count as evidence (slider default)
MP_DEFAULT_MIN_FRAMES = 2             # video: analysed frames in which the class must be seen to be 'Observed' (slider default)
MP_MIN_FRAMES_FOR_ABSENCE = 5         # fewer analysed frames than this -> an absence is 'Inconclusive', never 'Potentially Missing'
MP_MIN_DETECTIONS_FOR_ABSENCE = 5     # fewer detections in the whole video than this -> 'Inconclusive'
MP_MIN_RELIABLE_FRACTION = 0.30       # classifier confident on fewer crops than this -> it does not know this footage -> 'Inconclusive'
MP_NAME_MATCH_CUTOFF = 0.85           # spelling similarity needed to link a typed name to a classifier class name

OUTPUT_DIR = os.path.abspath("supermarket_ai_outputs")   # annotated videos + CSV files go here
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs folder:", OUTPUT_DIR)

## 4. Locate the model files
Uses files already in `/content` (or Drive) and only asks for an upload if something is missing.

In [ ]:
# Locate best.pt and grocery_classifier_1143.zip.
# 1) Looks in the usual Colab / project folders (so you do NOT have to upload again if they are already there).
# 2) If something is missing and we are in Colab, opens the upload dialog.
# 3) If you uploaded a file twice, Colab renames it "best (1).pt" - we simply use the newest match.

MOUNT_GOOGLE_DRIVE = False        # set True to mount Google Drive (Colab only) so files in MyDrive are found automatically
if MOUNT_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DETECTOR_PATH_OVERRIDE = None     # e.g. "/content/drive/MyDrive/best.pt"
CLASSIFIER_PATH_OVERRIDE = None   # e.g. "/content/drive/MyDrive/grocery_classifier_1143.zip"

SEARCH_DIRS = [os.getcwd(), "/content", "/content/drive/MyDrive", "/mnt/user-data/uploads", "/mnt/data"]

def find_newest(patterns):
    hits = []
    for d in SEARCH_DIRS:
        for p in patterns:
            hits += [f for f in glob.glob(os.path.join(d, p)) if os.path.isfile(f)]
    hits = sorted(set(hits), key=os.path.getmtime, reverse=True)
    return os.path.abspath(hits[0]) if hits else None

DETECTOR_PATTERNS = ["best.pt", "best (*).pt", "best*.pt"]
CLASSIFIER_PATTERNS = ["grocery_classifier*.zip", "grocery_classifier*.pt", "grocery_classifier*.pth"]

def resolve_assets():
    det = DETECTOR_PATH_OVERRIDE or find_newest(DETECTOR_PATTERNS)
    cls = CLASSIFIER_PATH_OVERRIDE or find_newest(CLASSIFIER_PATTERNS)
    return det, cls

WEIGHTS_PATH, CLASSIFIER_PATH = resolve_assets()

if (WEIGHTS_PATH is None or CLASSIFIER_PATH is None) and IN_COLAB:
    missing = [n for n, p in (("best.pt", WEIGHTS_PATH), ("grocery_classifier_1143.zip", CLASSIFIER_PATH)) if p is None]
    print("Missing:", missing, "-> please upload them now.")
    from google.colab import files
    files.upload()
    WEIGHTS_PATH, CLASSIFIER_PATH = resolve_assets()

problems = []
if WEIGHTS_PATH is None:
    problems.append("YOLOv8 detector weights (best.pt) not found. Upload it to /content or set DETECTOR_PATH_OVERRIDE.")
if CLASSIFIER_PATH is None:
    problems.append("Classifier checkpoint (grocery_classifier_1143.zip) not found. Upload it to /content or set CLASSIFIER_PATH_OVERRIDE.")
if problems:
    raise FileNotFoundError("\n".join(problems))

print("Detector   :", WEIGHTS_PATH, f"({os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB)")
print("Classifier :", CLASSIFIER_PATH, f"({os.path.getsize(CLASSIFIER_PATH)/1e6:.1f} MB)")
if CLASSIFIER_PATH.lower().endswith(".zip"):
    print("Classifier ZIP valid:", zipfile.is_zipfile(CLASSIFIER_PATH))

## 5. Load the YOLOv8 detector
Your trained `best.pt` — no pretrained substitute.

In [ ]:
# ---- YOLOv8 detector (your trained best.pt - NOT a pretrained substitute) ----
from ultralytics import YOLO

detector = YOLO(WEIGHTS_PATH)
print("Detector task :", detector.task)
print("Class names   :", detector.names)

if detector.task != "detect":
    raise RuntimeError(f"best.pt is a '{detector.task}' model, but this project needs a 'detect' model.")

def new_tracker_model():
    """Return a FRESH copy of the trained detector.

    ByteTrack keeps its state inside the model's predictor. Loading a fresh copy for every
    analysis guarantees track IDs start from scratch and never leak between videos.
    """
    return YOLO(WEIGHTS_PATH)

# NOTE: this detector has ONE class (usually 'object'): it finds product-like objects on shelves.
# Product identity comes from the classifier in the next cell, not from the detector.
DETECTOR_CLASS_NAMES = dict(detector.names)

## 6. Load the grocery classifier
The checkpoint is inspected first (state-dict vs. full model, keys, tensor shapes), the matching architecture is built, and the weights are loaded with `strict=True` so any mismatch fails loudly.

**To show real product names:** create `class_names.json` next to the notebook containing a JSON list of 25 names in training order (index 0 first) and re-run this cell.

In [ ]:
# ---- Grocery classifier: inspect checkpoint -> build matching architecture -> load ----
# Checkpoint inspection (kept from the original notebook): torch.load on the ZIP with weights_only=True.
if CLASSIFIER_PATH.lower().endswith(".zip"):
    with zipfile.ZipFile(CLASSIFIER_PATH) as z:
        _names_in_zip = z.namelist()
    print(f"ZIP contains {len(_names_in_zip)} entries, e.g.:", [n for n in _names_in_zip if not n.startswith("grocery_classifier/data/")][:6])

raw = torch.load(CLASSIFIER_PATH, map_location="cpu", weights_only=True)
print("Checkpoint type:", type(raw).__name__)

if not isinstance(raw, dict):
    raise TypeError(
        "The classifier file is a complete pickled model, not a state dictionary. "
        "This notebook only loads state dictionaries (safer). Re-save with torch.save(model.state_dict(), ...)."
    )

state = raw
for key in ("state_dict", "model_state_dict"):
    if key in raw and isinstance(raw[key], dict):
        state = raw[key]
        break
state = {(k[7:] if k.startswith("module.") else k): v for k, v in state.items()}   # strip DataParallel prefix

# --- Verify the architecture from the actual keys / shapes (we do not assume blindly) ---
tensor_keys = [k for k, v in state.items() if hasattr(v, "shape")]
has_resnet_stem = {"conv1.weight", "bn1.weight", "layer1.0.conv1.weight", "layer4.1.conv2.weight", "fc.weight", "fc.bias"} <= set(tensor_keys)
has_block3 = any(k.startswith("layer1.0.conv3") for k in tensor_keys)       # bottleneck blocks (ResNet50+) have conv3
has_layer4_block2 = any(k.startswith("layer4.2.") for k in tensor_keys)      # ResNet34 has 3 blocks in layer4
fc_out, fc_in = tuple(state["fc.weight"].shape)
print(f"conv1.weight {tuple(state['conv1.weight'].shape)} | fc.weight {tuple(state['fc.weight'].shape)}")

if not (has_resnet_stem and not has_block3 and not has_layer4_block2 and fc_in == 512 and state["conv1.weight"].shape[0] == 64):
    raise RuntimeError("Checkpoint does not match a ResNet18 layout. Keys/shapes printed above - adjust build code accordingly.")

NUM_CLASSES = int(fc_out)
classifier = torchvision.models.resnet18(weights=None, num_classes=NUM_CLASSES)
classifier.load_state_dict(state, strict=True)     # raises with exact key names if anything differs
classifier.to(DEVICE).eval()
CLASSIFIER_ARCH = "ResNet18"
print(f"Loaded {CLASSIFIER_ARCH} with {NUM_CLASSES} output classes (strict=True: every tensor matched).")

# --- Class labels: use a real mapping if one exists; never invent names ---
def load_class_names(n):
    # 1) names stored inside the checkpoint dictionary
    for key in ("class_names", "classes", "idx_to_class", "class_to_idx"):
        if key in raw:
            val = raw[key]
            if isinstance(val, (list, tuple)) and len(val) == n:
                return [str(v) for v in val], f"checkpoint key '{key}'"
            if isinstance(val, dict):
                if key == "class_to_idx":
                    val = {v: k for k, v in val.items()}
                try:
                    names = [str(val[i]) for i in range(n)]
                    return names, f"checkpoint key '{key}'"
                except Exception:
                    pass
    # 2) a mapping file next to the weights / in the working folders
    for d in SEARCH_DIRS + [os.path.dirname(CLASSIFIER_PATH)]:
        for fname in ("class_names.json", "classes.json", "class_names.txt", "classes.txt", "labels.txt"):
            p = os.path.join(d, fname)
            if os.path.isfile(p):
                try:
                    if p.endswith(".json"):
                        data = json.load(open(p))
                        names = [str(x) for x in data] if isinstance(data, list) else [str(data[str(i)]) for i in range(n)]
                    else:
                        names = [l.strip() for l in open(p) if l.strip()]
                    if len(names) == n:
                        return names, p
                    print(f"[!] {p} has {len(names)} names but the classifier has {n} outputs - ignoring it.")
                except Exception as e:
                    print(f"[!] Could not read {p}: {e}")
    return None, None

_names, _src = load_class_names(NUM_CLASSES)
if _names:
    CLASS_NAMES, CLASS_NAMES_SOURCE, CLASS_NAMES_AVAILABLE = _names, _src, True
else:
    CLASS_NAMES = [f"class_{i}" for i in range(NUM_CLASSES)]
    CLASS_NAMES_SOURCE, CLASS_NAMES_AVAILABLE = "not available", False
    print("\n[!] LIMITATION: the uploaded files contain no class-name mapping.")
    print(f"    Predictions are shown as class_0 ... class_{NUM_CLASSES-1}. To get real names, put a class_names.json")
    print(f"    (a list of {NUM_CLASSES} names in training order) next to the notebook and re-run this cell.")

CLASSIFIER_READY = True

@torch.no_grad()
def classify_crops(crops_bgr):
    """crops_bgr: list of HxWx3 uint8 BGR images -> list of (class_id, confidence)."""
    results = []
    for start in range(0, len(crops_bgr), CLS_BATCH):
        chunk = crops_bgr[start:start + CLS_BATCH]
        arr = np.stack([
            cv2.resize(cv2.cvtColor(c, cv2.COLOR_BGR2RGB), (CLS_INPUT_SIZE, CLS_INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
            for c in chunk
        ]).astype(np.float32) / 255.0
        arr = (arr - CLS_MEAN) / CLS_STD
        x = torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(DEVICE)
        probs = torch.softmax(classifier(x), dim=1)
        conf, idx = probs.max(dim=1)
        results += list(zip(idx.cpu().tolist(), conf.cpu().tolist()))
    return results

# Quick self-check on a blank image so a broken classifier fails HERE, not inside the dashboard.
_t = classify_crops([np.full((64, 48, 3), 127, np.uint8)])
assert len(_t) == 1 and 0 <= _t[0][0] < NUM_CLASSES, "classifier smoke test failed"
print("Classifier smoke test OK.")

## 7. Video pipeline: OpenCV → YOLOv8 → ByteTrack → classifier → CSV
`analyze_video()` is the single function the dashboard calls. The **frame interval** controls which frames are analysed (every frame is still written to the output video so its length and speed stay correct). ByteTrack state is created fresh for each video.

In [ ]:
# ---- Video pipeline: OpenCV read -> YOLOv8 detect -> ByteTrack -> classifier -> annotated video + CSV ----
#
# analyze_video(video_path, confidence, frame_interval, use_classifier, max_processed_frames, progress_callback)
#   returns (annotated_video_path, summary_markdown, results_dataframe, csv_path, stats_dict)
#
# Frame skipping: every frame IS read and WRITTEN to the output video (so timing and length are preserved),
# but only frames whose 0-based index is a multiple of `frame_interval` are sent to YOLO/ByteTrack/classifier.
# In between, the last analysed boxes are held on screen (thin outline) so the video does not flicker.

CSV_BASE_COLUMNS = ["frame_number", "timestamp_s", "track_id", "detector_class", "detector_confidence", "x1", "y1", "x2", "y2"]
CSV_CLASSIFIER_COLUMNS = ["classifier_class_id", "classifier_label", "classifier_confidence"]

def _track_color(tid):
    palette = [(46, 160, 22), (166, 130, 0), (90, 180, 255), (200, 100, 40), (60, 60, 200), (150, 60, 160), (20, 150, 200), (80, 190, 120)]
    return palette[int(tid) % len(palette)] if tid is not None else (120, 120, 120)

def _draw_overlay(frame, items, frame_idx, analysed_now, interval):
    h, w = frame.shape[:2]
    thick = 2 if analysed_now else 1
    dense = len(items) > 40
    scale = max(0.35, min(0.6, h / 1500.0))
    for (x1, y1, x2, y2, tid, label) in items:
        color = _track_color(tid)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick)
        if analysed_now and not (dense and tid is None):
            text = ("#" + str(tid) if tid is not None else "no-id")
            if label and not dense:
                text += " " + label
            cv2.putText(frame, text, (x1 + 2, max(10, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)
    hud = f"frame {frame_idx} | {'analysed' if analysed_now else 'held from last analysed frame'} | interval {interval}"
    (tw, th), _ = cv2.getTextSize(hud, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (0, 0), (tw + 12, th + 12), (24, 43, 34), -1)
    cv2.putText(frame, hud, (6, th + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return frame

def _to_browser_mp4(src, dst):
    """OpenCV's 'mp4v' codec does not play in most browsers; re-encode to H.264 if ffmpeg exists."""
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        return src, "ffmpeg not found - video may not play in the browser (CSV/table are unaffected)."
    cmd = [ffmpeg, "-y", "-loglevel", "error", "-i", src, "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p", "-movflags", "+faststart", dst]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0 and os.path.isfile(dst) and os.path.getsize(dst) > 0:
        os.remove(src)
        return dst, None
    return src, "ffmpeg re-encode failed: " + (r.stderr or "")[-300:]

def make_tracker_yaml(confidence):
    """Write a ByteTrack config whose thresholds follow the confidence slider.

    WHY: Ultralytics' bundled bytetrack.yaml has track_high_thresh = new_track_thresh = 0.25. Detections below 0.25 can
    never start a track and are not returned, so the old confidence slider did nothing between 0.10 and 0.25. Here the
    tracker thresholds are set to the chosen confidence, so the slider is honoured over its whole range.
    """
    try:
        base_path = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "trackers", TRACKER_CONFIG)
        with open(base_path) as f:
            cfg = yaml.safe_load(f)
        low_orig = float(cfg.get("track_low_thresh", 0.1))
        cfg["track_high_thresh"] = float(confidence)
        cfg["new_track_thresh"] = float(confidence)
        cfg["track_low_thresh"] = float(min(low_orig, confidence))
        path = os.path.join(OUTPUT_DIR, f"bytetrack_conf{int(round(float(confidence) * 100)):03d}.yaml")
        with open(path, "w") as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        return path, f"ByteTrack thresholds set to the confidence slider ({float(confidence):.2f})"
    except Exception as exc:                       # never block an analysis because of this convenience
        return TRACKER_CONFIG, f"could not customise ByteTrack thresholds ({type(exc).__name__}); using the bundled {TRACKER_CONFIG}"

def analyze_video(video_path, confidence=DEFAULT_CONF, frame_interval=DEFAULT_INTERVAL,
                  use_classifier=True, max_processed_frames=DEFAULT_MAX_PROCESSED, progress_callback=None):
    t_start = time.time()
    if not video_path or not os.path.isfile(str(video_path)):
        raise FileNotFoundError(f"Video file not found: {video_path!r}")
    confidence = float(confidence)
    frame_interval = int(frame_interval)
    if frame_interval not in VALID_INTERVALS:
        raise ValueError(f"frame_interval must be one of {VALID_INTERVALS}, got {frame_interval}")
    max_processed_frames = int(max_processed_frames or 0)
    use_classifier = bool(use_classifier) and bool(globals().get("CLASSIFIER_READY", False))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_path = os.path.join(OUTPUT_DIR, f"annotated_raw_{stamp}.mp4")
    final_path = os.path.join(OUTPUT_DIR, f"annotated_{stamp}.mp4")
    csv_path = os.path.join(OUTPUT_DIR, f"detections_{stamp}.csv")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open the video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps_assumed = not (fps and fps == fps and 0 < fps <= 240)
    if fps_assumed:
        fps = 25.0
    total_hint = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    tracker_model = new_tracker_model()          # fresh tracker state for this video
    tracker_cfg, tracker_note = make_tracker_yaml(confidence)
    writer = None
    size = None
    rows, frame_counts = [], []
    held_items = []
    frames_read = processed = 0
    truncated = False
    frame_idx = -1

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_idx += 1
            frames_read += 1
            if writer is None:
                size = (frame.shape[1], frame.shape[0])
                writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
                if not writer.isOpened():
                    raise RuntimeError("OpenCV VideoWriter could not be opened (mp4v).")
            elif (frame.shape[1], frame.shape[0]) != size:
                frame = cv2.resize(frame, size)

            analysed_now = (frame_idx % frame_interval == 0)
            if analysed_now:
                res = tracker_model.track(frame, persist=True, tracker=tracker_cfg, conf=confidence,
                                          imgsz=DETECTOR_IMGSZ, max_det=MAX_DET, device=INFER_DEVICE, verbose=False)[0]
                boxes = res.boxes
                dets = []
                if boxes is not None and len(boxes) > 0:
                    xyxy = boxes.xyxy.cpu().numpy()
                    confs = boxes.conf.cpu().numpy()
                    clss = boxes.cls.cpu().numpy().astype(int)
                    ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [None] * len(xyxy)
                    names = res.names
                    H, W = frame.shape[:2]
                    for (bx1, by1, bx2, by2), c, k, tid in zip(xyxy, confs, clss, ids):
                        x1, y1 = max(0, int(round(bx1))), max(0, int(round(by1)))
                        x2, y2 = min(W, int(round(bx2))), min(H, int(round(by2)))
                        dets.append(dict(x1=x1, y1=y1, x2=x2, y2=y2, conf=float(c),
                                         cls=str(names.get(int(k), int(k))) if isinstance(names, dict) else str(names[int(k)]),
                                         tid=(int(tid) if tid is not None else None)))
                # classifier on the crops of this frame (batched)
                preds = [None] * len(dets)
                if use_classifier and dets:
                    valid = [i for i, d in enumerate(dets) if (d["x2"] - d["x1"]) >= 2 and (d["y2"] - d["y1"]) >= 2]
                    out = classify_crops([frame[dets[i]["y1"]:dets[i]["y2"], dets[i]["x1"]:dets[i]["x2"]] for i in valid])
                    for i, p in zip(valid, out):
                        preds[i] = p
                held_items = []
                for d, p in zip(dets, preds):
                    row = {"frame_number": frame_idx, "timestamp_s": round(frame_idx / fps, 3), "track_id": d["tid"],
                           "detector_class": d["cls"], "detector_confidence": round(d["conf"], 4),
                           "x1": d["x1"], "y1": d["y1"], "x2": d["x2"], "y2": d["y2"]}
                    label = None
                    if use_classifier:
                        if p is None:
                            row.update(classifier_class_id=None, classifier_label=None, classifier_confidence=None)
                        else:
                            label = CLASS_NAMES[p[0]]
                            row.update(classifier_class_id=int(p[0]), classifier_label=label, classifier_confidence=round(float(p[1]), 4))
                    rows.append(row)
                    held_items.append((d["x1"], d["y1"], d["x2"], d["y2"], d["tid"], label))
                processed += 1
                frame_counts.append((frame_idx, len(dets)))
                if progress_callback:
                    frac = (frame_idx + 1) / total_hint if total_hint > 0 else 0.0
                    if max_processed_frames:
                        frac = max(frac, processed / max_processed_frames)
                    progress_callback(min(frac, 0.95), f"Analysed {processed} frames ({len(rows)} detections so far)")

            writer.write(_draw_overlay(frame, held_items, frame_idx, analysed_now, frame_interval))

            if max_processed_frames and processed >= max_processed_frames:
                truncated = True
                break
    finally:
        cap.release()                      # always release OpenCV resources
        if writer is not None:
            writer.release()

    if frames_read == 0:
        raise ValueError("The video contains no readable frames.")

    if progress_callback:
        progress_callback(0.97, "Encoding annotated video")
    video_out, video_warning = _to_browser_mp4(raw_path, final_path)

    columns = CSV_BASE_COLUMNS + (CSV_CLASSIFIER_COLUMNS if use_classifier else [])
    df = pd.DataFrame(rows, columns=columns)
    df["track_id"] = pd.array(df["track_id"], dtype="Int64")
    if use_classifier:
        df["classifier_class_id"] = pd.array(df["classifier_class_id"], dtype="Int64")
    df.to_csv(csv_path, index=False)       # header-only file when there are no detections

    unique_tracks = int(df["track_id"].dropna().nunique()) if len(df) else 0
    no_id = int(df["track_id"].isna().sum()) if len(df) else 0
    stats = dict(
        detections=int(len(df)), unique_tracks=unique_tracks, detections_without_track_id=no_id,
        processed_frames=int(processed), frames_read=int(frames_read), total_frames_reported=total_hint,
        fps=float(fps), fps_assumed=bool(fps_assumed), duration_s=round(frames_read / fps, 2),
        frame_interval=frame_interval, confidence=confidence, classifier_used=bool(use_classifier),
        truncated=bool(truncated), max_processed_frames=max_processed_frames, frame_counts=frame_counts,
        elapsed_s=round(time.time() - t_start, 1), video_warning=video_warning, width=size[0], height=size[1],
        tracker_note=tracker_note, video_path=str(video_path),
    )
    if use_classifier and len(df):
        stats["class_counts"] = df["classifier_label"].value_counts().to_dict()
    summary = build_summary_markdown(stats)
    if progress_callback:
        progress_callback(1.0, "Done")
    return video_out, summary, df, csv_path, stats

def build_summary_markdown(s):
    lines = ["### Analysis summary", ""]
    lines.append(f"- **Detection records:** {s['detections']:,}  *(one row per detected box in each analysed frame)*")
    lines.append(f"- **Unique ByteTrack IDs:** {s['unique_tracks']:,}  *(IDs can exceed the real number of objects if tracks are lost and re-created)*")
    if s["detections_without_track_id"]:
        lines.append(f"- **Detections without a track ID:** {s['detections_without_track_id']:,}")
    lines.append(f"- **Frames analysed:** {s['processed_frames']:,} of {s['frames_read']:,} frames read (frame interval {s['frame_interval']})")
    lines.append(f"- **Video:** {s['width']}×{s['height']}, {s['fps']:.2f} fps{' (assumed - file reported no fps)' if s['fps_assumed'] else ''}, {s['duration_s']} s covered")
    lines.append(f"- **Detection confidence threshold:** {s['confidence']:.2f}" + (f"  *({s['tracker_note']})*" if s.get('tracker_note') else ""))
    lines.append(f"- **Classifier:** {'on' if s['classifier_used'] else 'off'}" +
                 (" — labels are `class_N` because no class-name mapping was supplied" if s['classifier_used'] and not globals().get('CLASS_NAMES_AVAILABLE', False) else ""))
    lines.append(f"- **Processing time:** {s['elapsed_s']} s")
    if s["truncated"]:
        lines.append(f"- ⚠️ Stopped after the safety limit of {s['max_processed_frames']} analysed frames; the output video ends there.")
    if s["detections"] == 0:
        lines.append("- ℹ️ No objects were detected at this confidence. Try a lower threshold or a different video.")
    if s["frame_interval"] > 1:
        lines.append(f"- ℹ️ Trade-off: with interval {s['frame_interval']}, objects that appear and vanish between sampled frames are not seen, and ByteTrack must match boxes across larger gaps (more ID switches are possible).")
    if s["video_warning"]:
        lines.append(f"- ⚠️ {s['video_warning']}")
    return "\n".join(lines)

print("analyze_video() ready.")

## 8. Dashboard helpers
Cards, charts and tables built from the real analysis results (empty states before any analysis).

In [ ]:
# ---- Dashboard HTML helpers: every number below comes from the real analysis results ----
# (No sample/simulated data. Before an analysis, widgets show an empty state.)

def esc(x):
    return html.escape(str(x))

def kpi_card(label, value, foot, icon="◈", green=False):
    cls = "sai-kpi sai-kpi-green" if green else "sai-kpi"
    return (f'<div class="{cls}"><div class="sai-kpi-top"><span class="sai-kpi-icon">{icon}</span>'
            f'<span class="sai-kpi-label">{esc(label)}</span></div>'
            f'<div class="sai-kpi-value">{esc(value)}</div><div class="sai-kpi-foot">{esc(foot)}</div></div>')

def page_header(title, subtitle, pills=()):
    pill_html = "".join(f'<span class="sai-pill">{esc(p)}</span>' for p in pills)
    return (f'<div class="sai-header"><div><div class="sai-title">{esc(title)}</div>'
            f'<div class="sai-subtitle">{esc(subtitle)}</div></div><div class="sai-pills">{pill_html}</div></div>')

def empty_state(title, text):
    return f'<div class="sai-empty"><div class="sai-empty-title">{esc(title)}</div><div class="sai-empty-text">{esc(text)}</div></div>'

def fmt_int(n):
    return f"{int(n):,}"

# ---------- chart: detections per analysed frame (real data) ----------
def build_line_chart_html(stats):
    head = '<div class="sai-panel-title">Detections per analysed frame</div>'
    if not stats or not stats.get("frame_counts"):
        return head + empty_state("No analysis yet", "Run a video in AI Studio. The chart shows the real number of detections in each analysed frame.")
    pts = stats["frame_counts"]
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    W, H, L, R, T, B = 640, 220, 44, 14, 14, 30
    ymax = max(max(ys), 1)
    xmin, xmax = min(xs), max(xs)
    def X(x): return L + (W - L - R) * ((x - xmin) / (xmax - xmin) if xmax > xmin else 0.5)
    def Y(y): return T + (H - T - B) * (1 - y / ymax)
    line = " ".join(f"{X(x):.1f},{Y(y):.1f}" for x, y in pts)
    area = f"{X(xs[0]):.1f},{H-B} " + line + f" {X(xs[-1]):.1f},{H-B}"
    grid = ""
    for i in range(5):
        v = ymax * i / 4
        yy = Y(v)
        grid += f'<line x1="{L}" y1="{yy:.1f}" x2="{W-R}" y2="{yy:.1f}" stroke="#E3ECE6" stroke-width="1"/>'
        grid += f'<text x="{L-6}" y="{yy+4:.1f}" text-anchor="end" font-size="11" fill="#526358">{v:.0f}</text>'
    xlabels = (f'<text x="{L}" y="{H-8}" font-size="11" fill="#526358">frame {xmin}</text>'
               f'<text x="{W-R}" y="{H-8}" text-anchor="end" font-size="11" fill="#526358">frame {xmax}</text>')
    dots = "".join(f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="2.5" fill="#16834A"/>' for x, y in pts) if len(pts) <= 60 else ""
    svg = (f'<svg viewBox="0 0 {W} {H}" style="width:100%;height:auto;display:block" role="img" aria-label="Detections per analysed frame">'
           f'{grid}<polygon points="{area}" fill="#DDF3E4"/><polyline points="{line}" fill="none" stroke="#16834A" stroke-width="2.5" stroke-linejoin="round"/>{dots}{xlabels}</svg>')
    cap = f'<div class="sai-caption">{len(pts)} analysed frames · peak {ymax} detections · measured from this video</div>'
    return head + svg + cap

# ---------- donut: classifier label distribution (real data) ----------
def build_donut_html(df, stats):
    head = '<div class="sai-panel-title">Classifier predictions</div>'
    if not stats or not stats.get("classifier_used"):
        return head + empty_state("Classifier not used" if stats else "No analysis yet",
                                  "Enable the grocery classifier in AI Studio to see how predictions are distributed." if stats else
                                  "Run a video with the classifier enabled to see the distribution of predicted classes.")
    counts = stats.get("class_counts") or {}
    if not counts:
        return head + empty_state("No predictions", "No detections were classified in this run.")
    total = sum(counts.values())
    items = sorted(counts.items(), key=lambda kv: -kv[1])
    top = items[:5]
    other = sum(v for _, v in items[5:])
    if other:
        top.append(("other", other))
    colors = ["#16834A", "#2FB36D", "#7AD3A2", "#0F766E", "#5EC2B7", "#B7E4C7"]
    segs, acc = [], 0.0
    for i, (_, v) in enumerate(top):
        start, end = acc / total * 100, (acc + v) / total * 100
        segs.append(f"{colors[i % len(colors)]} {start:.2f}% {end:.2f}%")
        acc += v
    legend = "".join(
        f'<div class="sai-legend-row"><span class="sai-dot" style="background:{colors[i % len(colors)]}"></span>'
        f'<span class="sai-legend-name">{esc(n)}</span><span class="sai-legend-val">{v:,} ({v/total*100:.1f}%)</span></div>'
        for i, (n, v) in enumerate(top))
    donut = (f'<div class="sai-donut" style="background:conic-gradient({", ".join(segs)})"><div class="sai-donut-hole">'
             f'<div class="sai-donut-num">{total:,}</div><div class="sai-donut-lbl">classified</div></div></div>')
    note = "" if CLASS_NAMES_AVAILABLE else '<div class="sai-caption">Class names were not supplied, so labels are class_N.</div>'
    return head + f'<div class="sai-donut-wrap">{donut}<div class="sai-legend">{legend}</div></div>' + note

# ---------- table preview (real records) ----------
def build_recent_html(df, max_rows=8):
    head = '<div class="sai-panel-title">Latest detection records</div>'
    if df is None or len(df) == 0:
        return head + empty_state("No records yet", "Detection records from the latest analysis will be previewed here.")
    cols = [c for c in ["frame_number", "track_id", "detector_class", "detector_confidence", "classifier_label", "classifier_confidence"] if c in df.columns]
    show = df[cols].head(max_rows)
    th = "".join(f"<th>{esc(c.replace('_', ' '))}</th>" for c in cols)
    body = ""
    for _, r in show.iterrows():
        body += "<tr>" + "".join(f"<td>{'—' if pd.isna(r[c]) else esc(r[c])}</td>" for c in cols) + "</tr>"
    more = f'<div class="sai-caption">Showing the first {len(show)} of {len(df):,} records. Open Detection Results for the full table.</div>'
    return head + f'<div class="sai-table-wrap"><table class="sai-table"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table></div>' + more

def build_stats_strip(stats):
    if not stats:
        return empty_state("Awaiting analysis", "Detections, tracks and frames will appear here after you analyse a video.")
    cells = [("Detections", fmt_int(stats["detections"])), ("Unique track IDs", fmt_int(stats["unique_tracks"])),
             ("Frames analysed", fmt_int(stats["processed_frames"])), ("Frame interval", str(stats["frame_interval"]))]
    return '<div class="sai-strip">' + "".join(f'<div class="sai-strip-cell"><div class="sai-strip-val">{esc(v)}</div><div class="sai-strip-lbl">{esc(k)}</div></div>' for k, v in cells) + '</div>'

def status_text():
    return dict(
        detector=f"YOLOv8 · {os.path.basename(WEIGHTS_PATH)}",
        classifier=(f"{CLASSIFIER_ARCH} · {NUM_CLASSES} classes" if globals().get("CLASSIFIER_READY") else "Not loaded"),
    )
print("Dashboard helpers ready.")

## 8b. Missing Products engine
Pure comparison logic: reference-image analysis, typed-name matching, the Observed / Potentially Missing / Inconclusive rules, and evidence crops. It uses only the detector/classifier output — no product names are invented.

In [ ]:
# ---- Missing Products engine (pure logic + reference-image analysis + evidence crops) ----
# What this does, honestly:
#   * The YOLOv8 detector has ONE class ("object"): it finds product-like boxes but cannot name them.
#   * The grocery classifier assigns each crop to one of NUM_CLASSES trained classes. That is the ONLY product
#     identity the model can give, so every comparison below is at CLASSIFIER-CLASS level, not exact SKU level.
#   * "Expected" products come from (a) a reference shelf image (detector + classifier run on it) and/or
#     (b) a typed list (matched to classifier class names; "class_N" works even without a names file).
#   * "Observed" / "Potentially Missing" / "Inconclusive" are decided only from the real detection records of the video.

MP_STATUS_OBSERVED = "Observed"
MP_STATUS_MISSING = "Potentially Missing"
MP_STATUS_INCONCLUSIVE = "Inconclusive"
MP_STATUS_ORDER = {MP_STATUS_MISSING: 0, MP_STATUS_INCONCLUSIVE: 1, MP_STATUS_OBSERVED: 2}

MP_COLUMNS = ["product", "status", "source", "classifier_class_id", "reference_instances",
              "video_frames_confident", "video_tracks", "video_max_per_frame", "video_detections_any_confidence",
              "best_classifier_confidence", "first_seen_s", "last_seen_s", "evidence",
              "video_frames_analysed", "detection_confidence", "frame_interval",
              "min_classifier_confidence", "min_frames_required"]

REF_DET_COLUMNS = ["x1", "y1", "x2", "y2", "detector_confidence", "classifier_class_id", "classifier_label", "classifier_confidence"]

# ---------------------------------------------------------------- expected-product inputs
def normalise_name(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def parse_expected_text(text):
    """One product per line (';' also separates). A single line with commas is split on commas."""
    if text is None or not str(text).strip():
        return []
    raw = str(text).replace("\r", "\n")
    parts = re.split(r"[\n;]+", raw)
    if len([p for p in parts if p.strip()]) == 1 and "," in raw:
        parts = raw.split(",")
    out, seen = [], set()
    for p in parts:
        p = re.sub(r"^\s*(?:[-*\u2022]+|\d+[.)])\s*", "", p).strip()      # strip bullets / "1." / "2)"
        key = normalise_name(p)
        if p and key and key not in seen:
            seen.add(key)
            out.append(p)
    return out

def map_name_to_class(name, class_names, names_available):
    """Map a typed product name to a classifier class index. Returns (class_id or None, note). Never guesses."""
    n = normalise_name(name)
    m = re.fullmatch(r"class ?(\d+)", n)
    if m:                                                   # "class_7" / "class 7" always works
        k = int(m.group(1))
        if 0 <= k < len(class_names):
            return k, f"matched by class index ({class_names[k]})"
        return None, f"class index {k} is outside 0..{len(class_names) - 1}"
    if not names_available:
        return None, ("the classifier has no class-name mapping (its labels are class_0..class_%d), so a typed product name "
                      "cannot be linked to a class. Type 'class_N' or use a reference image instead" % (len(class_names) - 1))
    norm = [normalise_name(c) for c in class_names]
    exact = [i for i, c in enumerate(norm) if c == n]
    if len(exact) == 1:
        return exact[0], "exact class-name match"
    if len(exact) > 1:
        return None, "ambiguous: several classes share this name"
    padded = f" {n} "
    sub = [i for i, c in enumerate(norm) if c and len(n) >= 3 and (f" {c} " in padded or padded in f" {c} ")]
    if len(sub) == 1:
        return sub[0], f"partial name match -> '{class_names[sub[0]]}'"
    if len(sub) > 1:
        return None, "ambiguous: matches several classes (" + ", ".join(class_names[i] for i in sub[:5]) + ")"
    close = difflib.get_close_matches(n, norm, n=1, cutoff=MP_NAME_MATCH_CUTOFF)
    if close:
        i = norm.index(close[0])
        return i, f"close spelling match -> '{class_names[i]}'"
    return None, "no classifier class has this name, so the model cannot recognise it"

def expected_items_from_text(names, class_names, names_available):
    items = []
    for nm in names:
        cid, note = map_name_to_class(nm, class_names, names_available)
        items.append(dict(product=nm, source="typed list", class_id=cid, reference_instances=None,
                          ref_best_conf=None, reference_ok=True, note=note))
    return items

def expected_items_from_reference(ref_df, class_names, min_conf):
    """One expected item per classifier class seen in the reference image. Classes seen only with weak confidence are
    kept but flagged (reference_ok=False) so they can never produce a 'Potentially Missing' verdict."""
    items = []
    if ref_df is None or len(ref_df) == 0:
        return items
    ids = pd.to_numeric(ref_df["classifier_class_id"], errors="coerce").astype("float64")      # float: no <NA> in boolean masks
    confs = pd.to_numeric(ref_df["classifier_confidence"], errors="coerce").astype("float64")
    for cid in sorted(ids.dropna().astype(int).unique()):
        mask = ids == cid
        strong = int((mask & (confs >= min_conf)).sum())
        total = int(mask.sum())
        best = float(confs[mask].max())
        note = (f"{strong} of {total} reference crop(s) classified as this class with confidence >= {min_conf:.2f}" if strong
                else f"all {total} reference crop(s) of this class are below the reference confidence {min_conf:.2f} (best {best:.2f})")
        items.append(dict(product=class_names[cid], source="reference image", class_id=int(cid),
                          reference_instances=strong, ref_best_conf=best, reference_ok=strong > 0, note=note))
    return items

# ---------------------------------------------------------------- reference image analysis
def analyze_reference_image(image_rgb, confidence, min_ref_conf=None):
    """Detector + classifier on ONE image. Returns (records DataFrame, annotated RGB image, BGR original)."""
    min_ref_conf = MP_REF_MIN_CLS_CONF if min_ref_conf is None else min_ref_conf
    arr = np.asarray(image_rgb)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    bgr = cv2.cvtColor(np.ascontiguousarray(arr[..., :3].astype(np.uint8)), cv2.COLOR_RGB2BGR)
    H, W = bgr.shape[:2]
    res = detector.predict(bgr, conf=float(confidence), imgsz=DETECTOR_IMGSZ, max_det=MAX_DET,
                           device=INFER_DEVICE, verbose=False)[0]
    dets = []
    if res.boxes is not None and len(res.boxes) > 0:
        for (bx1, by1, bx2, by2), c in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()):
            x1, y1 = max(0, int(round(bx1))), max(0, int(round(by1)))
            x2, y2 = min(W, int(round(bx2))), min(H, int(round(by2)))
            if x2 - x1 >= 2 and y2 - y1 >= 2:
                dets.append((x1, y1, x2, y2, float(c)))
    preds = classify_crops([bgr[d[1]:d[3], d[0]:d[2]] for d in dets]) if dets else []
    rows = [dict(x1=d[0], y1=d[1], x2=d[2], y2=d[3], detector_confidence=round(d[4], 4),
                 classifier_class_id=int(p[0]), classifier_label=CLASS_NAMES[p[0]], classifier_confidence=round(float(p[1]), 4))
            for d, p in zip(dets, preds)]
    df = pd.DataFrame(rows, columns=REF_DET_COLUMNS)
    df["classifier_class_id"] = pd.array(df["classifier_class_id"], dtype="Int64")
    ann = bgr.copy()
    scale = max(0.35, min(0.7, H / 1400.0))
    for r in rows:
        strong = r["classifier_confidence"] >= min_ref_conf
        color = (46, 160, 22) if strong else (0, 160, 230)        # BGR: green = confident, amber = weak
        cv2.rectangle(ann, (r["x1"], r["y1"]), (r["x2"], r["y2"]), color, 2)
        if len(rows) <= 60:
            cv2.putText(ann, f"{r['classifier_label']} {r['classifier_confidence']:.2f}", (r["x1"] + 2, max(12, r["y1"] - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)
    return df, cv2.cvtColor(ann, cv2.COLOR_BGR2RGB), bgr

# ---------------------------------------------------------------- the comparison
def _top_classes_text(df, reliable, k=3):
    if not len(df) or not reliable.any() or "classifier_label" not in df.columns:
        return "none"
    vc = df.loc[reliable, "classifier_label"].value_counts().head(k)
    return ", ".join(f"{lbl} x{n}" for lbl, n in vc.items())

def compare_expected_vs_observed(expected, video_df, stats, class_names, min_cls_conf, min_frames,
                                 min_frames_for_absence=None, min_dets_for_absence=None, min_reliable_fraction=None):
    """Return a DataFrame (MP_COLUMNS) with one row per expected product. Uses ONLY the video's detection records."""
    min_frames_for_absence = MP_MIN_FRAMES_FOR_ABSENCE if min_frames_for_absence is None else min_frames_for_absence
    min_dets_for_absence = MP_MIN_DETECTIONS_FOR_ABSENCE if min_dets_for_absence is None else min_dets_for_absence
    min_reliable_fraction = MP_MIN_RELIABLE_FRACTION if min_reliable_fraction is None else min_reliable_fraction

    df = video_df.reset_index(drop=True) if video_df is not None else pd.DataFrame()
    n_frames = int(stats.get("processed_frames", 0))
    n_dets = len(df)
    if n_dets and "classifier_class_id" in df.columns:
        cid_all = pd.to_numeric(df["classifier_class_id"], errors="coerce").astype("float64")   # float: no <NA> in boolean masks
        conf_all = pd.to_numeric(df["classifier_confidence"], errors="coerce").astype("float64")
    else:
        cid_all = pd.Series([np.nan] * n_dets, dtype=float)
        conf_all = pd.Series([np.nan] * n_dets, dtype=float)
    reliable = cid_all.notna() & (conf_all >= min_cls_conf)
    reliable_frac = float(reliable.sum()) / n_dets if n_dets else 0.0
    fps = float(stats.get("fps", 0) or 0)
    ts = (lambda f: round(float(f) / fps, 2)) if fps else (lambda f: None)

    rows = []
    for it in expected:
        cid, label = it["class_id"], it["product"]
        base = dict(product=label, source=it["source"], classifier_class_id=cid, reference_instances=it["reference_instances"],
                    video_frames_confident=0, video_tracks=0, video_max_per_frame=0, video_detections_any_confidence=0,
                    best_classifier_confidence=None, first_seen_s=None, last_seen_s=None,
                    video_frames_analysed=n_frames, detection_confidence=stats.get("confidence"),
                    frame_interval=stats.get("frame_interval"), min_classifier_confidence=min_cls_conf, min_frames_required=min_frames)
        ref_txt = ""
        if it["source"] == "reference image":
            ref_txt = f" Reference image: {it['reference_instances']} confident instance(s)."

        if cid is None:
            base.update(status=MP_STATUS_INCONCLUSIVE,
                        evidence=f"Cannot be tested: '{label}' could not be linked to a classifier class - {it['note']}.")
            rows.append(base); continue
        if not it["reference_ok"]:
            m_any = (cid_all == cid)
            base.update(status=MP_STATUS_INCONCLUSIVE, video_detections_any_confidence=int(m_any.sum()),
                        evidence=f"Weak reference: {it['note']}. Not enough certainty that this product is really on the reference shelf, "
                                 f"so a missing/observed verdict would be unreliable. Video detections of this class (any confidence): {int(m_any.sum())}.")
            rows.append(base); continue

        m_any = (cid_all == cid)
        m_strong = m_any & (conf_all >= min_cls_conf)
        frames_strong = int(df.loc[m_strong, "frame_number"].nunique()) if n_dets else 0
        tracks = int(df.loc[m_strong, "track_id"].dropna().nunique()) if n_dets else 0
        per_frame = df.loc[m_strong].groupby("frame_number").size() if m_strong.any() else pd.Series(dtype=int)
        max_pf = int(per_frame.max()) if len(per_frame) else 0
        any_n = int(m_any.sum())
        any_frames = int(df.loc[m_any, "frame_number"].nunique()) if any_n else 0
        base.update(video_frames_confident=frames_strong, video_tracks=tracks, video_max_per_frame=max_pf,
                    video_detections_any_confidence=any_n)
        if m_strong.any():
            base.update(best_classifier_confidence=round(float(conf_all[m_strong].max()), 4),
                        first_seen_s=ts(df.loc[m_strong, "frame_number"].min()), last_seen_s=ts(df.loc[m_strong, "frame_number"].max()))
        elif any_n:
            base.update(best_classifier_confidence=round(float(conf_all[m_any].max()), 4))

        if frames_strong >= min_frames:
            ev = (f"Seen: crops classified as '{label}' with confidence >= {min_cls_conf:.2f} in {frames_strong} analysed frame(s) "
                  f"({base['first_seen_s']}s to {base['last_seen_s']}s), {tracks} ByteTrack ID(s), best confidence {base['best_classifier_confidence']:.2f}, "
                  f"up to {max_pf} at once in a single frame.{ref_txt}")
            if it["source"] == "reference image" and it["reference_instances"] and max_pf < it["reference_instances"]:
                ev += (f" Note: the reference image shows {it['reference_instances']} instance(s) but at most {max_pf} were confidently seen in one video frame "
                       f"(could be a different camera view, occlusion, or genuinely fewer items - not decided here).")
            base.update(status=MP_STATUS_OBSERVED, evidence=ev)
        elif any_n:
            base.update(status=MP_STATUS_INCONCLUSIVE,
                        evidence=(f"Weak evidence: {any_n} detection(s) in {any_frames} frame(s) were classified as '{label}', but only {frames_strong} "
                                  f"analysed frame(s) reached confidence >= {min_cls_conf:.2f} (need {min_frames}); best confidence {base['best_classifier_confidence']:.2f}. "
                                  f"Could be a real product seen briefly/partly, or a classifier confusion.{ref_txt}"))
        else:
            reasons = []
            if n_frames < min_frames_for_absence:
                reasons.append(f"only {n_frames} frame(s) were analysed (need >= {min_frames_for_absence} to judge absence)")
            if n_dets < min_dets_for_absence:
                reasons.append(f"the video produced only {n_dets} detection(s) in total (need >= {min_dets_for_absence}), so the shelf may not have been covered")
            if reliable_frac < min_reliable_fraction:
                reasons.append(f"the classifier was confident (>= {min_cls_conf:.2f}) on only {reliable_frac:.0%} of crops (need >= {min_reliable_fraction:.0%}), "
                               f"so it does not recognise this footage well enough for an absence to mean anything")
            if reasons:
                base.update(status=MP_STATUS_INCONCLUSIVE,
                            evidence=f"Not found, but this cannot be judged: " + "; ".join(reasons) + f".{ref_txt}")
            else:
                base.update(status=MP_STATUS_MISSING,
                            evidence=(f"No crop in {n_frames} analysed frame(s) ({n_dets} detections) was classified as '{label}' at any confidence. "
                                      f"The classifier was confident on {reliable_frac:.0%} of crops; most frequent confident classes: {_top_classes_text(df, reliable)}. "
                                      f"'Potentially' because the product may be hidden, out of camera view, undetected, or labelled as another class.{ref_txt}"))
        rows.append(base)

    out = pd.DataFrame(rows)
    for c in MP_COLUMNS:
        if c not in out.columns:
            out[c] = None
    out = out[MP_COLUMNS]
    out["_o"] = out["status"].map(MP_STATUS_ORDER)
    out = out.sort_values(["_o", "product"], kind="stable").drop(columns="_o").reset_index(drop=True)
    out["classifier_class_id"] = pd.array(out["classifier_class_id"], dtype="Int64")
    return out

# ---------------------------------------------------------------- evidence crops
def _save_crop(img_bgr, box, path, max_h=200):
    H, W = img_bgr.shape[:2]
    x1, y1, x2, y2 = box
    pad = 4
    crop = img_bgr[max(0, y1 - pad):min(H, y2 + pad), max(0, x1 - pad):min(W, x2 + pad)]
    if crop.size == 0:
        return None
    if crop.shape[0] > max_h:
        s = max_h / crop.shape[0]
        crop = cv2.resize(crop, (max(1, int(crop.shape[1] * s)), max_h), interpolation=cv2.INTER_AREA)
    return path if cv2.imwrite(path, crop) else None

def build_evidence_gallery(video_path, video_df, comp_df, ref_bgr, ref_df, min_cls_conf, max_products=10):
    """Real crops only: best reference crop and best video crop per product. Returns [(image_path, caption), ...]."""
    ev_dir = os.path.join(OUTPUT_DIR, "evidence")
    os.makedirs(ev_dir, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    cid_all = pd.to_numeric(video_df["classifier_class_id"], errors="coerce").astype("float64") if video_df is not None and len(video_df) else pd.Series(dtype=float)
    conf_all = pd.to_numeric(video_df["classifier_confidence"], errors="coerce").astype("float64") if video_df is not None and len(video_df) else pd.Series(dtype=float)
    plan, wanted = [], {}
    for k, r in comp_df.iterrows():
        if k >= max_products:
            break
        if pd.isna(r["classifier_class_id"]):
            continue
        cid = int(r["classifier_class_id"])
        ref_item = None
        if ref_bgr is not None and ref_df is not None and len(ref_df):
            g = ref_df[pd.to_numeric(ref_df["classifier_class_id"], errors="coerce").astype("float64") == cid]
            if len(g):
                b = g.sort_values("classifier_confidence", ascending=False).iloc[0]
                ref_item = (int(b.x1), int(b.y1), int(b.x2), int(b.y2), float(b.classifier_confidence))
        vid_item = None
        if len(cid_all):
            m = (cid_all == cid) & conf_all.notna()
            if m.any():
                b = video_df.loc[conf_all[m].idxmax()]
                vid_item = (int(b["frame_number"]), (int(b["x1"]), int(b["y1"]), int(b["x2"]), int(b["y2"])), float(b["classifier_confidence"]), b["track_id"])
                wanted.setdefault(vid_item[0], []).append(k)
        plan.append((k, r["product"], r["status"], ref_item, vid_item))
    vid_paths = {}
    if wanted and video_path and os.path.isfile(str(video_path)):
        cap = cv2.VideoCapture(str(video_path))
        needed = sorted(wanted)
        ptr, idx = 0, -1
        try:
            while ptr < len(needed) and cap.grab():          # sequential grab = exact frame numbers (no seeking drift)
                idx += 1
                if idx == needed[ptr]:
                    ok, frame = cap.retrieve()
                    if ok:
                        for k in wanted[idx]:
                            item = next(p for p in plan if p[0] == k)[4]
                            vid_paths[k] = _save_crop(frame, item[1], os.path.join(ev_dir, f"{stamp}_video_{k}.png"))
                    ptr += 1
        finally:
            cap.release()
    gallery = []
    for k, product, status, ref_item, vid_item in plan:
        if ref_item is not None:
            p = _save_crop(ref_bgr, ref_item[:4], os.path.join(ev_dir, f"{stamp}_ref_{k}.png"))
            if p:
                gallery.append((p, f"REFERENCE | {product} | classifier conf {ref_item[4]:.2f}"))
        if vid_item is not None and vid_paths.get(k):
            tid = "-" if pd.isna(vid_item[3]) else f"#{int(vid_item[3])}"
            gallery.append((vid_paths[k], f"VIDEO ({status}) | {product} | frame {vid_item[0]} | track {tid} | conf {vid_item[2]:.2f}"))
    return gallery

# ---------------------------------------------------------------- observed products (needs NO baseline)
OBS_COLUMNS = ["product", "classifier_class_id", "status", "detections", "frames_seen", "frames_confident", "video_tracks",
               "best_classifier_confidence", "mean_classifier_confidence", "first_seen_s", "last_seen_s", "note"]
OBS_CONFIDENT = "Detected"
OBS_WEAK = "Detected - weak evidence"

def summarize_observed_products(video_df, stats, class_names, min_cls_conf, min_frames):
    """Aggregate the video's real detection records per classifier class. Repeated detections of one physical item are
    collapsed by reporting distinct frames and distinct ByteTrack IDs next to the raw detection count (a count is NOT a stock count)."""
    if video_df is None or len(video_df) == 0 or "classifier_class_id" not in video_df.columns:
        return pd.DataFrame(columns=OBS_COLUMNS)
    df = video_df.reset_index(drop=True)
    cid, conf = _num(df["classifier_class_id"]), _num(df["classifier_confidence"])
    fps = float((stats or {}).get("fps", 0) or 0)
    rows = []
    for k in sorted(cid.dropna().astype(int).unique()):
        m = cid == k
        ms = m & (conf >= min_cls_conf)
        f_conf = int(df.loc[ms, "frame_number"].nunique())
        ok = f_conf >= min_frames
        fs = df.loc[ms, "frame_number"] if ms.any() else df.loc[m, "frame_number"]
        rows.append(dict(
            product=class_names[k], classifier_class_id=int(k), status=OBS_CONFIDENT if ok else OBS_WEAK, detections=int(m.sum()),
            frames_seen=int(df.loc[m, "frame_number"].nunique()), frames_confident=f_conf,
            video_tracks=int(df.loc[m, "track_id"].dropna().nunique()),
            best_classifier_confidence=round(float(conf[m].max()), 4), mean_classifier_confidence=round(float(conf[m].mean()), 4),
            first_seen_s=(round(float(fs.min()) / fps, 2) if fps else None), last_seen_s=(round(float(fs.max()) / fps, 2) if fps else None),
            note=("" if ok else f"only {f_conf} frame(s) with classifier confidence >= {min_cls_conf:.2f} (need {min_frames})")))
    out = pd.DataFrame(rows, columns=OBS_COLUMNS)
    out["_o"] = (out["status"] != OBS_CONFIDENT).astype(int)
    out = out.sort_values(["_o", "detections"], ascending=[True, False], kind="stable").drop(columns="_o").reset_index(drop=True)
    out["classifier_class_id"] = pd.array(out["classifier_class_id"], dtype="Int64")
    return out

# ---------------------------------------------------------------- diagnostics (real counts / confidences from THIS run; no paths or file names)
MP_DEBUG = True      # True = also print a '[MissingProducts]' diagnostics block to the notebook output on every run

def _num(series):
    """Numeric float64 view (pandas NA -> NaN) so boolean masks never contain <NA>."""
    return pd.to_numeric(series, errors="coerce").astype("float64")

def _conf_stats(values):
    s = _num(pd.Series(values)).dropna()
    if not len(s):
        return None
    return dict(min=round(float(s.min()), 3), median=round(float(s.median()), 3), max=round(float(s.max()), 3))

def _top_labels(df, mask=None, k=8):
    if df is None or not len(df) or "classifier_label" not in df.columns:
        return {}
    d = df if mask is None else df[mask]
    return {str(a): int(b) for a, b in d["classifier_label"].dropna().value_counts().head(k).items()}

def _suggest_threshold(median):
    return max(0.30, math.floor(float(median) * 20) / 20)

def build_mp_diagnostics(stage, ref_given, ref_df, video_df, stats, comp, confidence, min_cls_conf, min_frames, typed_count=0, n_expected=0,
                         mode=None, video_received=None, observed_df=None, stop_reason=None):
    """Summarise what each pipeline stage actually produced, plus plain-language hints derived only from those numbers."""
    d = dict(stage=stage, mode=mode, stop_reason=stop_reason,
             inputs=dict(video_received=video_received, reference_image_received=bool(ref_given), typed_names=int(typed_count)),
             thresholds=dict(detection_confidence=confidence, classifier_confidence=min_cls_conf, frames_required=min_frames),
             typed_names=int(typed_count), expected_products=int(n_expected),
             observed_classes=(None if observed_df is None else int(len(observed_df))),
             comparison_rows=(None if comp is None else int(len(comp))))
    hints = []
    if ref_given:
        n = 0 if ref_df is None else len(ref_df)
        c = _num(ref_df["classifier_confidence"]) if n else pd.Series(dtype="float64")
        d["reference"] = dict(detections=int(n), classifier_confidence=_conf_stats(c), at_or_above_threshold=int((c >= min_cls_conf).sum()),
                              discarded_below_threshold=int((c < min_cls_conf).sum()), classes=_top_labels(ref_df))
        if n == 0:
            hints.append(f"The detector found no objects in the reference image at detection confidence {confidence:.2f}. Lower 'Detection confidence', use a closer / larger photo of the shelf, or type the expected products manually.")
        elif d["reference"]["at_or_above_threshold"] == 0:
            st = d["reference"]["classifier_confidence"]
            hints.append(f"All {n} reference crops were classified below {min_cls_conf:.2f} (highest {st['max']:.2f}, median {st['median']:.2f}). "
                         f"Lower 'Classifier confidence needed as evidence' (for example to {_suggest_threshold(st['median']):.2f}); the classifier may not recognise shelf crops well.")
    if video_df is not None:
        n = len(video_df)
        c = _num(video_df["classifier_confidence"]) if n and "classifier_confidence" in video_df.columns else pd.Series(dtype="float64")
        ok = (c >= min_cls_conf)
        s = stats or {}
        d["video"] = dict(frames_sampled=int(s.get("processed_frames", 0)), frames_read=int(s.get("frames_read", 0)),
                          detections_at_detection_threshold=int(n), tracks=int(s.get("unique_tracks", 0)), without_track_id=int(s.get("detections_without_track_id", 0)),
                          classified=int(c.notna().sum()), classifier_confidence=_conf_stats(c), kept_at_classifier_threshold=int(ok.sum()),
                          discarded_below_classifier_threshold=int(n - ok.sum()),
                          classes_detected=_top_labels(video_df), confident_classes=_top_labels(video_df, mask=ok) if n else {})
        if n == 0:
            hints.append(f"The detector found no objects in the {d['video']['frames_sampled']} analysed video frames at detection confidence {confidence:.2f}. "
                         f"This is NOT evidence that products are missing. Lower 'Detection confidence', use a smaller frame interval, or check the video.")
        elif int(ok.sum()) == 0:
            st = d["video"]["classifier_confidence"]
            hints.append(f"None of the {n} video detections reached classifier confidence {min_cls_conf:.2f} (highest {st['max']:.2f}, median {st['median']:.2f}). "
                         f"Lower 'Classifier confidence needed as evidence' (for example to {_suggest_threshold(st['median']):.2f}).")
        elif d["video"]["frames_sampled"] < MP_MIN_FRAMES_FOR_ABSENCE:
            hints.append(f"Only {d['video']['frames_sampled']} frames were analysed; use a smaller frame interval or a longer video before trusting an absence.")
    if comp is not None and len(comp):
        d["comparison"] = [dict(product=r["product"], status=r["status"], class_id=(None if pd.isna(r["classifier_class_id"]) else int(r["classifier_class_id"])),
                                source=r["source"], frames_confident=int(r["video_frames_confident"]), detections_any_conf=int(r["video_detections_any_confidence"]))
                           for _, r in comp.iterrows()]
    d["hints"] = hints
    return d

def print_mp_diagnostics(d):
    print("[MissingProducts] diagnostics\n" + json.dumps(d, indent=1, default=str))

print("Missing-products engine ready.")


In [ ]:
# ---- Missing Products: HTML helpers (all numbers/labels come from the comparison table) ----
MP_BADGE_CLASS = {MP_STATUS_OBSERVED: "sai-st-observed", MP_STATUS_MISSING: "sai-st-missing", MP_STATUS_INCONCLUSIVE: "sai-st-inconclusive"}

def mp_status_badge(status):
    return f'<span class="sai-status {MP_BADGE_CLASS.get(status, "sai-st-inconclusive")}">{esc(status)}</span>'

def mp_limitations_html():
    if CLASS_NAMES_AVAILABLE:
        names_line = f"Class names come from <b>{esc(os.path.basename(str(CLASS_NAMES_SOURCE)))}</b>, so typed product names are matched against them."
    else:
        names_line = ("<b>No class-name mapping is available</b>, so typed product names cannot be matched to the classifier. "
                      "Use a <b>reference shelf image</b>, or type <b>class_N</b> (see the class list below).")
    inner = f"""
    <div class="sai-panel-title">How this check works - and what it cannot do</div>
    <ul class="sai-list">
      <li>The YOLOv8 detector has a single class (<b>{esc(', '.join(map(str, DETECTOR_CLASS_NAMES.values())))}</b>): it finds product-like boxes but cannot name them.</li>
      <li>Product identity comes only from the grocery classifier, which can output <b>{NUM_CLASSES} classes</b>. The comparison is therefore at <b>classifier-class level</b>, not exact product (SKU) level. Two different products of the same class look identical to this check, and a product outside those classes is forced into the nearest one.</li>
      <li>{names_line}</li>
      <li>A reference image or expected list is <b>optional</b>. Without one, the page lists the products detected in the video, but <b>cannot say what is missing</b> because there is nothing to compare against.</li>
      <li><b>Potentially Missing</b> is never a proof of absence: the product may be hidden, outside the camera view, undetected, or labelled as a different class. When the evidence is too thin to judge, the result is <b>Inconclusive</b> instead.</li>
    </ul>"""
    return card(inner)

def mp_class_list_html():
    chips = "".join(f'<span class="sai-chip">{i} · {esc(n)}</span>' for i, n in enumerate(CLASS_NAMES))
    src = f"loaded from {esc(os.path.basename(str(CLASS_NAMES_SOURCE)))}" if CLASS_NAMES_AVAILABLE else "no names file found - generic labels"
    return card(f'<div class="sai-panel-title">Classes the model can recognise ({NUM_CLASSES})</div>'
                f'<div class="sai-caption" style="margin:0 0 8px">Labels {src}. Only these can be compared.</div><div class="sai-chips">{chips}</div>')

def mp_summary_html(comp_df, meta=None):
    if comp_df is None:
        return empty_state("No comparison yet", "Upload a shelf video and press Compare. The reference image and expected-product list are optional: without them, the products detected in the video are listed.")
    counts = comp_df["status"].value_counts().to_dict() if len(comp_df) else {}
    cards = (
        '<div class="sai-mp-kpis">'
        + kpi_card("Observed", counts.get(MP_STATUS_OBSERVED, 0), "Seen in the video with enough confidence", "✓", green=True)
        + kpi_card("Potentially Missing", counts.get(MP_STATUS_MISSING, 0), "Expected, not found in the video", "!")
        + kpi_card("Inconclusive", counts.get(MP_STATUS_INCONCLUSIVE, 0), "Not enough evidence to decide", "?")
        + kpi_card("Expected products", len(comp_df), "Checked at classifier-class level", "▤")
        + '</div>')
    notes = ""
    if meta:
        bits = [f"video analysed: {meta.get('frames_analysed', '?')} frames (interval {meta.get('interval', '?')}), detection confidence {meta.get('conf', 0):.2f}",
                f"classifier confidence needed: {meta.get('min_cls_conf', 0):.2f}", f"frames needed to confirm: {meta.get('min_frames', '?')}"]
        if meta.get("reused"):
            bits.append("re-used an existing analysis (no re-run)")
        notes = f'<div class="sai-caption" style="margin:8px 2px 0">{esc(" · ".join(bits))}</div>'
        for w in meta.get("warnings", []):
            notes += f'<div class="sai-warnline">⚠ {esc(w)}</div>'
    return cards + notes

def mp_results_html(comp_df):
    head = '<div class="sai-panel-title">Expected vs. detected products</div>'
    if comp_df is None or len(comp_df) == 0:
        return card(head + empty_state("No results", "Nothing to compare yet."))
    body = ""
    for _, r in comp_df.iterrows():
        cls_txt = "—" if pd.isna(r["classifier_class_id"]) else f"class {int(r['classifier_class_id'])}"
        ref_txt = "—" if r["reference_instances"] is None or pd.isna(r["reference_instances"]) else str(int(r["reference_instances"]))
        body += (f'<tr><td><b>{esc(r["product"])}</b><div class="sai-caption" style="margin:2px 0 0">{esc(r["source"])} · {esc(cls_txt)}</div></td>'
                 f'<td>{mp_status_badge(r["status"])}</td><td class="sai-num">{ref_txt}</td>'
                 f'<td class="sai-num">{int(r["video_frames_confident"])}</td><td class="sai-num">{int(r["video_tracks"])}</td>'
                 f'<td class="sai-evidence">{esc(r["evidence"])}</td></tr>')
    th = "".join(f"<th>{h}</th>" for h in ["Product", "Status", "In reference", "Frames seen", "Track IDs", "Evidence"])
    return card(head + f'<div class="sai-table-wrap"><table class="sai-table"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table></div>')

def mp_message_html(title, text):
    return card(empty_state(title, text))

def mp_observed_html(obs, title="Products detected in the video"):
    head = f'<div class="sai-panel-title">{esc(title)}</div>'
    cap = ('<div class="sai-caption" style="margin:0 0 8px">Each row is a <b>classifier class</b>, not an exact product. “Detections” counts boxes over all analysed frames, '
           'so one physical item seen in many frames is counted many times; use <b>Frames seen</b> and <b>Track IDs</b> as the better guide.</div>')
    if obs is None or len(obs) == 0:
        return card(head + empty_state("Nothing to list", "No classified detections."))
    body = ""
    for _, r in obs.iterrows():
        badge = mp_status_badge(MP_STATUS_OBSERVED) if r["status"] == OBS_CONFIDENT else mp_status_badge(MP_STATUS_INCONCLUSIVE)
        body += (f'<tr><td><b>{esc(r["product"])}</b><div class="sai-caption" style="margin:2px 0 0">class {int(r["classifier_class_id"])}</div></td>'
                 f'<td>{badge}<div class="sai-caption" style="margin:2px 0 0">{esc(r["status"])}</div></td>'
                 f'<td class="sai-num">{int(r["detections"])}</td><td class="sai-num">{int(r["frames_seen"])}</td><td class="sai-num">{int(r["video_tracks"])}</td>'
                 f'<td class="sai-num">{float(r["best_classifier_confidence"]):.2f}</td><td class="sai-num">{float(r["mean_classifier_confidence"]):.2f}</td>'
                 f'<td class="sai-evidence">{esc(r["note"])}</td></tr>')
    th = "".join(f"<th>{h}</th>" for h in ["Product class", "Status", "Detections", "Frames seen", "Track IDs", "Best conf", "Mean conf", "Note"])
    return card(head + cap + f'<div class="sai-table-wrap"><table class="sai-table"><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table></div>')

def mp_observed_summary_html(obs, meta, reason):
    n_conf = int((obs["status"] == OBS_CONFIDENT).sum()) if len(obs) else 0
    cards = ('<div class="sai-mp-kpis">'
             + kpi_card("Product classes detected", len(obs), "Classifier classes seen in the video", "▤", green=True)
             + kpi_card("Confident classes", n_conf, f"Seen in at least {meta.get('min_frames', '?')} frames", "✓")
             + kpi_card("Detections", int(obs["detections"].sum()) if len(obs) else 0, "Boxes over all analysed frames", "▦")
             + kpi_card("Frames analysed", meta.get("frames_analysed", "?"), f"Interval {meta.get('interval', '?')}", "◉") + '</div>')
    notes = f'<div class="sai-warnline">ℹ {esc(reason)}</div>'
    for w in meta.get("warnings", []):
        notes += f'<div class="sai-warnline">⚠ {esc(w)}</div>'
    return cards + notes

def mp_diagnostics_html(d):
    """Diagnostics panel built from build_mp_diagnostics(): real counts and confidences from this run."""
    if not d:
        return ""
    rows = []
    def row(k, v):
        rows.append(f'<tr><td><b>{esc(k)}</b></td><td>{esc(v)}</td></tr>')
    def cs(s):
        return "none" if not s else f"min {s['min']:.2f} · median {s['median']:.2f} · max {s['max']:.2f}"
    t, inp = d["thresholds"], d.get("inputs", {})
    row("Stage reached", d["stage"] + (f" - {d['stop_reason']}" if d.get("stop_reason") else ""))
    row("Mode", d.get("mode") or "not decided")
    row("Inputs received", f"shelf video: {inp.get('video_received')} · reference image: {inp.get('reference_image_received')} · typed names: {inp.get('typed_names')}")
    row("Thresholds used", f"detection {t['detection_confidence']:.2f} · classifier {t['classifier_confidence']:.2f} · frames needed {t['frames_required']}")
    row("Products", f"expected: {d['expected_products']} · classes observed in video: {d.get('observed_classes')} · comparison rows: {d.get('comparison_rows')}")
    if "reference" in d:
        r = d["reference"]
        row("Reference image", f"{r['detections']} detections · classifier confidence {cs(r['classifier_confidence'])} · {r['at_or_above_threshold']} kept, {r['discarded_below_threshold']} below threshold")
        if r["classes"]:
            row("Reference classes", ", ".join(f"{k} x{v}" for k, v in r["classes"].items()))
    if "video" in d:
        v = d["video"]
        row("Video frames", f"{v['frames_sampled']} sampled/processed (of {v['frames_read']} read)")
        row("Video detections", f"{v['detections_at_detection_threshold']} at the detection threshold ({v['tracks']} track IDs, {v['without_track_id']} without ID) · "
                                f"{v['kept_at_classifier_threshold']} kept / {v['discarded_below_classifier_threshold']} below the classifier threshold")
        row("Video classifier confidence", cs(v["classifier_confidence"]))
        if v["classes_detected"]:
            row("Classes detected (all confidences)", ", ".join(f"{k} x{n}" for k, n in v["classes_detected"].items()))
    hints = "".join(f"<li>{esc(h)}</li>" for h in d.get("hints", []))
    hint_html = f'<div class="sai-h3">What to try</div><ul class="sai-list">{hints}</ul>' if hints else ""
    return card('<div class="sai-panel-title">Diagnostics (real model output from this run)</div>'
                f'<div class="sai-table-wrap"><table class="sai-table"><tbody>{"".join(rows)}</tbody></table></div>' + hint_html)

print("Missing-products HTML helpers ready.")


## 9. Build the dashboard
Readable-by-construction styling: a custom theme sets every text/label/table/radio colour, dark-mode colours are forced identical to light mode, and the CSS only targets our own `sai-*` classes.

In [ ]:
# ---- Supermarket AI dashboard (Gradio 6) ----
# READABILITY STRATEGY (fixes the old "white text on white" problem):
#   1. A custom Gradio theme sets text, label, input, table, radio and button colours, and the *_dark
#      variants are set to the SAME light values -> dark mode can never turn text white on white cards.
#   2. A tiny script in <head> also removes the 'dark' class Gradio may add.
#   3. The CSS below only styles our own classes (sai-*). There is no broad `.gradio-container *` rule.

PALETTE = dict(bg="#F4F7F5", card="#FFFFFF", text="#1B2B22", muted="#526358", heading="#183B2A",
               green="#16834A", green_dark="#106A3C", soft="#E5F4E9", border="#DCE6DF")

def make_theme():
    base = gr.themes.Base(primary_hue="emerald", secondary_hue="teal", neutral_hue="slate",
                          font=["Inter", "Segoe UI", "Arial", "sans-serif"],
                          radius_size=gr.themes.sizes.radius_lg)
    p = PALETTE
    light = dict(
        body_background_fill=p["bg"], background_fill_primary=p["card"], background_fill_secondary=p["bg"],
        body_text_color=p["text"], body_text_color_subdued=p["muted"],
        block_background_fill=p["card"], block_border_color=p["border"], block_border_width="1px",
        block_label_background_fill=p["card"], block_label_text_color=p["heading"], block_label_border_color=p["border"],
        block_title_text_color=p["heading"], block_title_background_fill="transparent",
        block_info_text_color=p["muted"], panel_background_fill=p["card"], panel_border_color=p["border"],
        border_color_primary=p["border"], border_color_accent=p["green"], color_accent_soft=p["soft"],
        input_background_fill=p["card"], input_background_fill_focus=p["card"], input_background_fill_hover=p["card"],
        input_border_color=p["border"], input_border_color_focus=p["green"], input_placeholder_color=p["muted"],
        slider_color=p["green"],
        table_text_color=p["text"], table_even_background_fill=p["card"], table_odd_background_fill="#F7FAF8",
        table_border_color=p["border"], table_row_focus=p["soft"],
        checkbox_label_background_fill=p["card"], checkbox_label_background_fill_hover=p["soft"],
        checkbox_label_background_fill_selected=p["soft"], checkbox_label_border_color=p["border"],
        checkbox_label_border_color_hover=p["green"], checkbox_label_border_color_selected=p["green"],
        checkbox_label_text_color=p["text"], checkbox_label_text_color_selected=p["heading"],
        button_primary_background_fill=p["green"], button_primary_background_fill_hover=p["green_dark"],
        button_primary_border_color=p["green"], button_primary_border_color_hover=p["green_dark"],
        button_primary_text_color="#FFFFFF", button_primary_text_color_hover="#FFFFFF",
        button_secondary_background_fill=p["card"], button_secondary_background_fill_hover=p["soft"],
        button_secondary_border_color=p["border"], button_secondary_border_color_hover=p["green"],
        button_secondary_text_color=p["text"], button_secondary_text_color_hover=p["heading"],
        link_text_color=p["green"], link_text_color_hover=p["green_dark"],
    )
    allowed = set(inspect.signature(gr.themes.Base.set).parameters)
    kw = {}
    for k, v in light.items():
        if k in allowed:
            kw[k] = v
        if k + "_dark" in allowed:
            kw[k + "_dark"] = v          # dark mode = identical light colours
    return base.set(**kw)

HEAD = """<script>
(function () {
  function forceLight() {
    document.documentElement.classList.remove('dark');
    if (document.body) document.body.classList.remove('dark');
  }
  forceLight();
  document.addEventListener('DOMContentLoaded', function () {
    forceLight();
    var o = new MutationObserver(forceLight);
    o.observe(document.documentElement, {attributes: true, attributeFilter: ['class']});
    if (document.body) o.observe(document.body, {attributes: true, attributeFilter: ['class']});
  });
})();
</script>"""

CSS = """
:root { color-scheme: light; }
.gradio-container { max-width: 1500px !important; margin: 0 auto !important; background: #F4F7F5 !important; }

/* ---------- layout ---------- */
#sai-sidebar { background: #FFFFFF; border: 1px solid #DCE6DF; border-radius: 16px; padding: 18px 14px; gap: 6px;
               align-self: flex-start; box-shadow: 0 1px 2px rgba(24,59,42,.04); }
.sai-brand { display: flex; align-items: center; gap: 10px; padding: 2px 6px 14px; }
.sai-logo { width: 34px; height: 34px; border-radius: 10px; background: #16834A; color: #FFFFFF; font-weight: 800;
            display: flex; align-items: center; justify-content: center; font-size: 17px; }
.sai-brand-name { color: #183B2A; font-size: 18px; font-weight: 800; line-height: 1.1; }
.sai-brand-cap { color: #526358; font-size: 10px; letter-spacing: 1.2px; margin-top: 2px; }
.sai-menu-label { color: #526358; font-size: 11px; font-weight: 700; letter-spacing: 1px; padding: 6px 8px 2px; }
.sai-nav, .sai-nav button { justify-content: flex-start !important; text-align: left !important; width: 100%; font-weight: 600 !important; }
.sai-sidebox { background: #E5F4E9; border: 1px solid #CFE8D6; border-radius: 12px; padding: 12px 14px; margin-top: 12px;
               color: #183B2A; font-size: 12px; line-height: 1.7; }
.sai-sidebox b { color: #183B2A; }

/* ---------- headers ---------- */
.sai-header { display: flex; justify-content: space-between; align-items: flex-start; gap: 12px; flex-wrap: wrap; padding: 4px 2px 6px; }
.sai-title { color: #183B2A; font-size: 28px; font-weight: 800; line-height: 1.2; }
.sai-subtitle { color: #526358; font-size: 14px; margin-top: 4px; }
.sai-pills { display: flex; gap: 8px; flex-wrap: wrap; }
.sai-pill { background: #FFFFFF; border: 1px solid #DCE6DF; color: #1B2B22; border-radius: 999px; padding: 5px 12px; font-size: 12px; font-weight: 600; }

/* ---------- cards ---------- */
.sai-card { background: #FFFFFF; border: 1px solid #DCE6DF; border-radius: 16px; padding: 18px; color: #1B2B22;
            box-shadow: 0 1px 2px rgba(24,59,42,.04); height: 100%; box-sizing: border-box; }
.sai-panel-title { color: #183B2A; font-size: 16px; font-weight: 750; margin-bottom: 10px; }
.sai-caption { color: #526358; font-size: 12px; margin-top: 8px; }
.sai-kpi { background: #FFFFFF; border: 1px solid #DCE6DF; border-radius: 16px; padding: 16px 18px; min-height: 124px; box-sizing: border-box;
           box-shadow: 0 1px 2px rgba(24,59,42,.04); }
.sai-kpi-top { display: flex; align-items: center; gap: 10px; }
.sai-kpi-icon { width: 30px; height: 30px; border-radius: 50%; background: #E5F4E9; color: #0F6B3A; display: inline-flex;
                align-items: center; justify-content: center; font-size: 14px; font-weight: 700; }
.sai-kpi-label { color: #526358; font-size: 12px; font-weight: 700; letter-spacing: .3px; }
.sai-kpi-value { color: #183B2A; font-size: 30px; font-weight: 800; margin: 12px 0 4px; line-height: 1.1; }
.sai-kpi-foot { color: #526358; font-size: 12px; }
.sai-kpi-green { background: #16834A; border-color: #16834A; }
.sai-kpi-green .sai-kpi-icon { background: rgba(255,255,255,.22); color: #FFFFFF; }
.sai-kpi-green .sai-kpi-label, .sai-kpi-green .sai-kpi-foot { color: #FFFFFF; }
.sai-kpi-green .sai-kpi-value { color: #FFFFFF; }

.sai-banner { background: #183B2A; border-radius: 16px; padding: 18px 22px; }
.sai-banner-title { color: #FFFFFF; font-size: 20px; font-weight: 800; }
.sai-banner-text { color: #DDF0E3; font-size: 13px; margin-top: 4px; }

.sai-empty { background: #F7FAF8; border: 1px dashed #C9D8CE; border-radius: 12px; padding: 22px 16px; text-align: center; }
.sai-empty-title { color: #183B2A; font-weight: 700; font-size: 14px; }
.sai-empty-text { color: #526358; font-size: 12.5px; margin-top: 4px; }

/* ---------- pipeline steps ---------- */
.sai-step { display: flex; align-items: center; gap: 12px; padding: 9px 0; border-bottom: 1px solid #EEF3EF; }
.sai-step:last-child { border-bottom: none; }
.sai-step-n { width: 28px; height: 28px; border-radius: 50%; background: #E5F4E9; color: #0F6B3A; font-weight: 800; font-size: 12px;
              display: flex; align-items: center; justify-content: center; flex: none; }
.sai-step-name { color: #183B2A; font-weight: 700; font-size: 14px; }
.sai-step-det { color: #526358; font-size: 12px; }
.sai-step-main { flex: 1; min-width: 0; }
.sai-badge { background: #E5F4E9; color: #14532D; border-radius: 999px; padding: 3px 10px; font-size: 11px; font-weight: 700; white-space: nowrap; }
.sai-badge-warn { background: #FFF4D6; color: #6B4E00; }

/* ---------- tables / strips / donut ---------- */
.sai-table-wrap { overflow-x: auto; }
.sai-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.sai-table th { background: #EAF2EC; color: #183B2A; text-align: left; padding: 8px 10px; font-weight: 700; text-transform: capitalize; white-space: nowrap; }
.sai-table td { color: #1B2B22; padding: 8px 10px; border-bottom: 1px solid #EEF3EF; background: #FFFFFF; }
.sai-strip { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; }
.sai-strip-cell { background: #F4F9F5; border: 1px solid #DCE6DF; border-radius: 12px; padding: 10px 12px; }
.sai-strip-val { color: #183B2A; font-size: 22px; font-weight: 800; }
.sai-strip-lbl { color: #526358; font-size: 11.5px; }
.sai-donut-wrap { display: flex; align-items: center; gap: 18px; flex-wrap: wrap; }
.sai-donut { width: 150px; height: 150px; border-radius: 50%; position: relative; flex: none; }
.sai-donut-hole { position: absolute; inset: 24px; background: #FFFFFF; border-radius: 50%; display: flex; flex-direction: column; align-items: center; justify-content: center; }
.sai-donut-num { color: #183B2A; font-weight: 800; font-size: 20px; }
.sai-donut-lbl { color: #526358; font-size: 11px; }
.sai-legend { flex: 1; min-width: 160px; }
.sai-legend-row { display: flex; align-items: center; gap: 8px; font-size: 12.5px; padding: 3px 0; }
.sai-dot { width: 10px; height: 10px; border-radius: 50%; flex: none; }
.sai-legend-name { color: #1B2B22; flex: 1; }
.sai-legend-val { color: #526358; }
.sai-list { color: #1B2B22; font-size: 13.5px; line-height: 1.7; margin: 6px 0 0; padding-left: 20px; }
.sai-list li { color: #1B2B22; }
.sai-h3 { color: #183B2A; font-weight: 750; font-size: 15px; margin: 14px 0 4px; }
.sai-p { color: #1B2B22; font-size: 13.5px; line-height: 1.65; margin: 4px 0; }

/* ---------- controls (only cosmetic sizing here; colours come from the theme) ---------- */
.sai-slider button { width: 26px !important; min-width: 26px !important; height: 26px !important; min-height: 26px !important; padding: 2px !important; }
.sai-radio label.selected { background: #E5F4E9 !important; border-color: #16834A !important; color: #183B2A !important; }
.sai-radio label.selected span { color: #183B2A !important; font-weight: 700; }
.sai-radio input[type="radio"] { accent-color: #16834A; }

/* ---------- Missing Products ---------- */
.sai-mp-kpis { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
@media (max-width: 900px) { .sai-mp-kpis { grid-template-columns: repeat(2, 1fr); } .sai-strip { grid-template-columns: repeat(2, 1fr); } }
.sai-status { display: inline-block; border-radius: 999px; padding: 3px 11px; font-size: 12px; font-weight: 700; white-space: nowrap; border: 1px solid transparent; }
.sai-st-observed { background: #E5F4E9; color: #14532D; border-color: #BFE3CA; }
.sai-st-missing { background: #FDECEA; color: #8A1C12; border-color: #F5C2BC; }
.sai-st-inconclusive { background: #FFF4D6; color: #6B4E00; border-color: #F1DC9B; }
.sai-evidence { color: #1B2B22; font-size: 12.5px; line-height: 1.5; min-width: 320px; }
.sai-num { text-align: center; white-space: nowrap; }
.sai-warnline { background: #FFF4D6; color: #6B4E00; border: 1px solid #F1DC9B; border-radius: 10px; padding: 8px 12px; font-size: 12.5px; margin-top: 8px; }
.sai-chips { display: flex; flex-wrap: wrap; gap: 6px; }
.sai-chip { background: #F4F9F5; border: 1px solid #DCE6DF; color: #183B2A; border-radius: 999px; padding: 3px 10px; font-size: 12px; }
"""

def make_html(value, extra_class=""):
    return gr.HTML(value, elem_classes=["sai-html"] + ([extra_class] if extra_class else []))

def card(inner):
    return f'<div class="sai-card">{inner}</div>'

def pipeline_html():
    st = status_text()
    steps = [
        ("01", "OpenCV", f"Video reading, frame handling, annotation (cv2 {cv2.__version__})", "Ready", False),
        ("02", "YOLOv8 detector", f"{st['detector']} · classes: {', '.join(map(str, DETECTOR_CLASS_NAMES.values()))}", "Loaded", False),
        ("03", "ByteTrack", f"Ultralytics tracker · {TRACKER_CONFIG}", "Configured", False),
        ("04", "Grocery classifier", st["classifier"] + ("" if CLASS_NAMES_AVAILABLE else " · labels unavailable"),
         "Loaded" if globals().get("CLASSIFIER_READY") else "Missing", not CLASS_NAMES_AVAILABLE),
        ("05", "Results & CSV export", "Aggregated table + downloadable CSV", "Ready", False),
        ("06", "Missing-product check", "Reference image / product list vs. detected classifier classes", "Class-level", not CLASS_NAMES_AVAILABLE),
    ]
    rows = "".join(
        f'<div class="sai-step"><div class="sai-step-n">{n}</div><div class="sai-step-main"><div class="sai-step-name">{esc(name)}</div>'
        f'<div class="sai-step-det">{esc(det)}</div></div><span class="sai-badge{" sai-badge-warn" if warn else ""}">{esc(badge)}</span></div>'
        for n, name, det, badge, warn in steps)
    return card('<div class="sai-panel-title">Pipeline overview</div>' + rows)

def about_html():
    st = status_text()
    labels_line = (f"Class names loaded from {CLASS_NAMES_SOURCE}." if CLASS_NAMES_AVAILABLE else
                   "No class-name mapping was supplied, so predictions are shown as class_0 … class_%d." % (NUM_CLASSES - 1))
    inner = f"""
    <div class="sai-panel-title">What this prototype actually does</div>
    <div class="sai-h3">1 · OpenCV — video reading and annotation</div>
    <div class="sai-p">Reads the uploaded or recorded video frame by frame, draws boxes, track IDs and labels, and writes the annotated video. All frames are written, so the output keeps the original length and frame rate.</div>
    <div class="sai-h3">2 · YOLOv8 — object detection</div>
    <div class="sai-p">Your trained detector ({esc(st['detector'])}) finds objects in every analysed frame. It has a single class ({esc(', '.join(map(str, DETECTOR_CLASS_NAMES.values())))}): it locates product-like items, it does not name them.</div>
    <div class="sai-h3">3 · ByteTrack — object tracking</div>
    <div class="sai-p">Links detections across analysed frames and gives each object a track ID. A track ID is not a product identity, and IDs can change if an object is lost and found again.</div>
    <div class="sai-h3">4 · Grocery classifier — per-crop classification</div>
    <div class="sai-p">A {esc(CLASSIFIER_ARCH)} with {NUM_CLASSES} output classes classifies each detected crop (resized to {CLS_INPUT_SIZE}×{CLS_INPUT_SIZE}, ImageNet normalisation — assumed, because the checkpoint does not store its transform). {esc(labels_line)}</div>
    <div class="sai-h3">5 · Results aggregation and CSV export</div>
    <div class="sai-p">Every detection becomes one row (frame number, timestamp, track ID, detector class and confidence, box, classifier prediction and confidence). The same table is shown in the dashboard and saved as a CSV.</div>
    <div class="sai-h3">6 · Missing-product comparison</div>
    <div class="sai-p">Expected products come from a reference shelf image (detector + classifier run on it) and/or a typed list. They are compared with the classifier classes found in the video's detection records. A class seen with enough confidence in enough frames is <b>Observed</b>; a class never detected, when the video gives enough evidence to judge, is <b>Potentially Missing</b>; everything else is <b>Inconclusive</b>. The comparison is class-level, not exact-product.</div>
    <div class="sai-h3">Frame-processing modes</div>
    <div class="sai-p">Interval 1 analyses every frame; 2, 3 and 5 analyse every 2nd, 3rd or 5th frame. Larger intervals are faster but can miss objects that appear between sampled frames and make tracking harder.</div>
    """
    return card(inner)

def limits_html():
    inner = """
    <div class="sai-panel-title">Implemented vs. not implemented</div>
    <div class="sai-h3">Implemented</div>
    <ul class="sai-list"><li>Detection, tracking, per-crop classification, annotated video and CSV export.</li>
    <li>Missing Products: class-level comparison of expected vs. detected products, with evidence crops and CSV export.</li>
    <li>Confidence threshold and frame interval that directly control the real pipeline.</li></ul>
    <div class="sai-h3">Not implemented (possible future work)</div>
    <ul class="sai-list"><li>Inventory counting, stock levels, sales analytics or automatic restocking.</li>
    <li>Product (SKU) recognition by the tracker, and verified classifier accuracy on this footage.</li>
    <li>Live camera streaming (the webcam input records a clip first).</li>
    <li>Exact-product (SKU) matching and shelf-position / planogram checks: the model can only name the classes its classifier was trained on.</li></ul>
    <div class="sai-h3">Known limitations</div>
    <ul class="sai-list"><li>Classifier labels need the original training class mapping.</li>
    <li>Confidence scores are not the same as accuracy.</li></ul>
    """
    return card(inner)

DISPLAY_ROW_LIMIT = 5000

def to_display_df(df):
    """Gradio cannot JSON-serialise pandas NA values, so convert them to blanks for display only."""
    if df is None:
        df = pd.DataFrame(columns=CSV_BASE_COLUMNS)
    shown = df.head(DISPLAY_ROW_LIMIT).astype(object)
    return shown.where(df.head(DISPLAY_ROW_LIMIT).notna(), "")

N_OUTPUTS = 15   # annotated video, summary, studio table, studio csv, results table, results csv, 4 KPI cards, studio strip, chart, donut, recent table, results note

def make_outputs(annotated, summary_md, df, csv_path, stats, status="Ready", status_foot="Detector + classifier loaded"):
    has = stats is not None
    note = ("Run an analysis in AI Studio to populate this table." if not has else
            f"{len(df):,} records from the latest analysis." + (f" Table shows the first {DISPLAY_ROW_LIMIT:,}; the CSV contains all of them." if len(df) > DISPLAY_ROW_LIMIT else "")
            if len(df) else "The latest analysis found no detections, so the table and CSV contain headers only.")
    table = to_display_df(df)
    out = (
        annotated, summary_md, table, csv_path, table, csv_path,
        kpi_card("Object detections", fmt_int(stats["detections"]) if has else "—", "Records across analysed frames" if has else "Run an analysis to populate", "▣"),
        kpi_card("Unique tracked objects", fmt_int(stats["unique_tracks"]) if has else "—", "Distinct ByteTrack IDs" if has else "Run an analysis to populate", "◎"),
        kpi_card("Frames processed", fmt_int(stats["processed_frames"]) if has else "—",
                 (f"of {fmt_int(stats['frames_read'])} read · interval {stats['frame_interval']}" if has else "Not available yet"), "▤"),
        kpi_card("Pipeline status", status, status_foot, "✓", green=True),
        build_stats_strip(stats),
        card(build_line_chart_html(stats)),
        card(build_donut_html(df, stats)),
        card(build_recent_html(df)),
        make_note_html(note),
    )
    assert len(out) == N_OUTPUTS, f"make_outputs returned {len(out)} values, expected {N_OUTPUTS}"
    return out

def make_note_html(text):
    return f'<div class="sai-caption" style="margin:2px 2px 8px">{esc(text)}</div>'

def run_analysis(video_path, confidence, frame_interval, use_classifier, max_frames, last_state=None, progress=gr.Progress()):
    """Gradio callback. Inputs (6): video, confidence, frame interval, classifier on/off, frame limit, previous analysis (State).
    Returns the 15 dashboard outputs plus the updated analysis State (re-usable by the Missing Products page)."""
    if not video_path:
        return make_outputs(None, "### Video required\n\nUpload a video or record a webcam clip first.", None, None, None, "Waiting", "No video selected") + (last_state,)
    try:
        annotated, summary, df, csv_path, stats = analyze_video(
            video_path, float(confidence), int(frame_interval), bool(use_classifier), int(max_frames),
            progress_callback=lambda f, d: progress(f, desc=d))
        state = dict(video_path=str(video_path), df=df, stats=stats, annotated=annotated)
        return make_outputs(annotated, summary, df, csv_path, stats, "Complete", f"{stats['processed_frames']} frames analysed in {stats['elapsed_s']} s") + (state,)
    except Exception as exc:
        traceback.print_exc()          # full traceback goes to the notebook output - nothing is hidden
        msg = f"### Analysis failed\n\n`{type(exc).__name__}: {exc}`\n\nThe full traceback is printed in the notebook output below the launch cell."
        return make_outputs(None, msg, None, None, None, "Error", type(exc).__name__) + (last_state,)

def clear_results():
    return make_outputs(None, "### Analysis summary\n\nResults cleared. Upload a video and press **Analyze video**.", None, None, None) + (None,)


# ======================= Missing Products callbacks =======================
def _mp_display(df, columns=None):
    """Gradio cannot JSON-serialise pandas NA values -> blanks for display only."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=columns or MP_COLUMNS)
    shown = df.astype(object)
    return shown.where(df.notna(), "")

def mp_placeholder():
    ex = [CLASS_NAMES[0]] + ([CLASS_NAMES[1]] if NUM_CLASSES > 1 else [])
    return "\n".join(ex)          # real entries of the classifier's own vocabulary - nothing invented

def mp_outputs(summary, results, table, csv, gallery, ref_img, video, state):
    return (summary, results, table, csv, gallery, ref_img, video, state)

def _mp_scrub(msg):
    """Remove file-system paths from error text before it is shown or printed."""
    return re.sub(r"(?:[A-Za-z]:)?(?:[\\/][^\s:'\"]+)+", "<path>", str(msg))

def run_missing_products(ref_image, expected_text, video_path, reuse_last, confidence, frame_interval, max_frames,
                         min_cls_conf, min_frames, last_state=None, progress=gr.Progress()):
    """Gradio callback. Inputs (10): reference image, typed list, video, reuse checkbox, detection confidence, frame interval,
    frame limit, classifier confidence, frames required, previous analysis (State). Returns 8 outputs (MP_N_OUTPUTS).

    Two independent workflows - the video is ALWAYS analysed:
      A) no baseline (no reference image, no usable typed list): list the products actually detected; no missing-product verdict.
      B) baseline (reference image and/or typed list): compare expected vs. detected -> Observed / Potentially Missing / Inconclusive,
         and also list any other classes detected in the video.
    "Nothing to compare against" is never reported as "nothing detected"."""
    stage = "reading the inputs"
    typed, expected = [], []
    ref_df = ref_ann = ref_bgr = video_df = stats = obs = comp = None
    have_ref = ref_image is not None
    video_received, mode = bool(video_path), None

    def diag(label, stop_reason=None):
        return build_mp_diagnostics(label, have_ref, ref_df, video_df, stats, comp, float(confidence), float(min_cls_conf), int(min_frames),
                                    typed_count=len(typed), n_expected=len(expected), mode=mode, video_received=video_received,
                                    observed_df=obs, stop_reason=stop_reason)
    def fail(title, text, ref_img=None, d=None):
        if d is not None and MP_DEBUG:
            print_mp_diagnostics(d)
        return mp_outputs(mp_message_html(title, text), mp_diagnostics_html(d) if d else mp_results_html(None),
                          _mp_display(None), None, [], ref_img, None, last_state)
    try:
        confidence, frame_interval = float(confidence), int(frame_interval)
        max_frames, min_cls_conf, min_frames = int(max_frames), float(min_cls_conf), int(min_frames)
        if not globals().get("CLASSIFIER_READY", False):
            return fail("Classifier not loaded", "The Missing Products check needs the grocery classifier. Run the classifier cell first.")
        typed = parse_expected_text(expected_text)
        reuse_ok = bool(reuse_last) and bool(last_state) and last_state.get("df") is not None and bool(last_state["stats"].get("classifier_used"))
        video_received = bool(video_path) or reuse_ok
        if not reuse_ok and not video_path:
            return fail("Shelf video needed", "Upload or record the current shelf video." +
                        (" The re-use option found no earlier analysis that used the classifier." if reuse_last else "") +
                        " The reference image and expected-product list are optional.", None, diag("stopped: no shelf video", "no video was received"))

        warnings = []
        stage = "analysing the reference image"
        progress(0.02, desc="Analysing reference image" if have_ref else "Reading expected products")
        if have_ref:
            ref_df, ref_ann, ref_bgr = analyze_reference_image(ref_image, confidence, min_cls_conf)
            expected += expected_items_from_reference(ref_df, CLASS_NAMES, min_cls_conf)
        stage = "building the expected-product list"
        by_class = {e["class_id"]: e for e in expected if e["class_id"] is not None}
        for item in expected_items_from_text(typed, CLASS_NAMES, CLASS_NAMES_AVAILABLE):
            if item["class_id"] in by_class:
                by_class[item["class_id"]]["source"] = "reference image + typed list"
            else:
                expected.append(item)
        mode = "comparison (baseline available)" if expected else "observed products only (no baseline)"
        if expected:
            reason = ""
        elif have_ref:
            reason = (f"The reference image produced no usable detections at confidence {confidence:.2f}, and no expected products were typed, so there is no baseline. "
                      "Type the expected products manually or lower 'Detection confidence'. Showing what the video contains instead.")
        else:
            reason = ("No reference image or expected-product list was given, so missing products cannot be determined reliably. "
                      "Showing the products the model detected in the video; add a reference image or expected list to enable the comparison.")

        stage = "analysing the shelf video"
        if reuse_ok:
            video_df, stats, vpath, annotated, reused = last_state["df"], last_state["stats"], last_state["video_path"], last_state.get("annotated"), True
            new_state = last_state
            if abs(float(stats["confidence"]) - confidence) > 1e-9:
                warnings.append(f"The re-used video analysis used detection confidence {stats['confidence']:.2f}; the reference image used {confidence:.2f}.")
        else:
            def cb(frac, desc):
                progress(0.05 + 0.80 * float(frac), desc=desc)
            annotated, _summary_md, video_df, _det_csv, stats = analyze_video(
                video_path, confidence, frame_interval, True, max_frames, progress_callback=cb)
            vpath, reused = str(video_path), False
            new_state = dict(video_path=vpath, df=video_df, stats=stats, annotated=annotated)
        if stats.get("truncated"):
            warnings.append(f"The video analysis stopped at the safety limit of {stats['max_processed_frames']} analysed frames, so later parts of the video were not checked.")
        if stats.get("video_warning"):
            warnings.append(stats["video_warning"])

        stage = "summarising the products observed in the video"
        obs = summarize_observed_products(video_df, stats, CLASS_NAMES, min_cls_conf, min_frames)
        meta = dict(frames_analysed=stats["processed_frames"], interval=stats["frame_interval"], conf=float(stats["confidence"]),
                    min_cls_conf=min_cls_conf, min_frames=min_frames, reused=reused, warnings=warnings)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        if not expected:                                                   # ---- Workflow A: observed products only
            if len(video_df) == 0:
                d = diag("completed", "the detector found no objects in the analysed video frames")
                if MP_DEBUG:
                    print_mp_diagnostics(d)
                return mp_outputs(mp_message_html("No products detected in the video",
                                                  f"The detector found no objects in {stats['processed_frames']} analysed frames at detection confidence {confidence:.2f}. "
                                                  "That is not evidence that products are missing. " + reason),
                                  mp_diagnostics_html(d), _mp_display(None, OBS_COLUMNS), None, [], ref_ann, annotated, new_state)
            stage = "collecting evidence crops"
            try:
                gal_df = obs.head(10).assign(status="Detected")[["product", "status", "classifier_class_id"]]
                gallery = build_evidence_gallery(vpath, video_df, gal_df, None, None, min_cls_conf)
            except Exception as gexc:
                traceback.print_exc(); gallery = []
                warnings.append(f"Evidence crops could not be created ({type(gexc).__name__}: {gexc}); the table is unaffected.")
            csv_path = os.path.join(OUTPUT_DIR, f"products_detected_{stamp}.csv")
            obs.to_csv(csv_path, index=False)
            d = diag("completed")
            if MP_DEBUG:
                print_mp_diagnostics(d)
            progress(1.0, desc="Done")
            return mp_outputs(mp_observed_summary_html(obs, meta, reason), mp_observed_html(obs) + mp_diagnostics_html(d),
                              _mp_display(obs, OBS_COLUMNS), csv_path, gallery, ref_ann, annotated, new_state)

        # ---- Workflow B: expected vs. detected
        stage = "comparing expected vs. detected"
        progress(0.88, desc="Comparing expected vs. detected")
        comp = compare_expected_vs_observed(expected, video_df, stats, CLASS_NAMES, min_cls_conf, min_frames)
        if len(video_df) == 0:
            warnings.append("The detector found no objects in the analysed video frames, so every product is reported as Inconclusive (see diagnostics).")
        stage = "collecting evidence crops"
        progress(0.93, desc="Collecting evidence crops")
        try:
            gallery = build_evidence_gallery(vpath, video_df, comp, ref_bgr, ref_df, min_cls_conf)
        except Exception as gexc:
            traceback.print_exc(); gallery = []
            warnings.append(f"Evidence crops could not be created ({type(gexc).__name__}: {gexc}); the verdicts above are unaffected.")
        stage = "writing the CSV"
        csv_path = os.path.join(OUTPUT_DIR, f"missing_products_{stamp}.csv")
        comp.to_csv(csv_path, index=False)
        exp_ids = {e["class_id"] for e in expected if e["class_id"] is not None}
        other = obs[~obs["classifier_class_id"].astype("float64").isin(exp_ids)].reset_index(drop=True) if len(obs) else obs
        d = diag("completed")
        if MP_DEBUG:
            print_mp_diagnostics(d)
        progress(1.0, desc="Done")
        results = mp_results_html(comp)
        if len(other):
            results += mp_observed_html(other, "Other classes detected in the video (not in the expected list)")
        return mp_outputs(mp_summary_html(comp, meta), results + mp_diagnostics_html(d), _mp_display(comp), csv_path,
                          gallery, ref_ann, annotated, new_state)
    except Exception as exc:
        traceback.print_exc()
        try:
            d = diag(f"failed while {stage}", f"{type(exc).__name__}: {_mp_scrub(exc)}")
        except Exception:
            d = None
        return fail("Comparison failed", f"Stopped while {stage}: {type(exc).__name__}: {_mp_scrub(exc)} (full traceback is printed in the notebook output).", ref_ann, d)

def clear_missing():
    return mp_outputs(mp_summary_html(None), mp_results_html(None), _mp_display(None), None, [], None, None, None)

MP_N_OUTPUTS = 8

PAGE_OVERVIEW, PAGE_STUDIO, PAGE_MISSING, PAGE_RESULTS, PAGE_ABOUT = range(5)
NAV_ITEMS = [("▦  Dashboard Overview", PAGE_OVERVIEW), ("◉  AI Studio", PAGE_STUDIO), ("◌  Missing Products", PAGE_MISSING),
             ("▤  Detection Results", PAGE_RESULTS), ("ⓘ  About the Pipeline", PAGE_ABOUT)]

def build_dashboard():
    # Gradio 6 moved theme/css/head from Blocks() to launch(); older versions still expect them in Blocks().
    launch_takes_style = "css" in inspect.signature(gr.Blocks.launch).parameters
    theme = make_theme()
    blocks_kw = {} if launch_takes_style else dict(theme=theme, css=CSS, head=HEAD)
    launch_kw = dict(theme=theme, css=CSS, head=HEAD) if launch_takes_style else {}
    st = status_text()
    device_pill = "GPU" if DEVICE.type == "cuda" else "CPU"

    with gr.Blocks(title="Supermarket AI | Retail Analytics", **blocks_kw) as demo:
        analysis_state = gr.State(None)     # latest analysis (video path, records, stats) - lets Missing Products re-use it
        with gr.Row(equal_height=False):
            # ---------------- sidebar ----------------
            with gr.Column(scale=0, min_width=235, elem_id="sai-sidebar"):
                make_html('<div class="sai-brand"><div class="sai-logo">S</div><div><div class="sai-brand-name">Supermarket AI</div>'
                          '<div class="sai-brand-cap">RETAIL VISION</div></div></div>')
                make_html('<div class="sai-menu-label">MENU</div>')
                nav_btns = [gr.Button(label, variant=("primary" if i == 0 else "secondary"), elem_classes=["sai-nav"]) for label, i in NAV_ITEMS]
                make_html(f'<div class="sai-sidebox"><b>Models loaded</b><br>Detector: {esc(st["detector"])}<br>Classifier: {esc(st["classifier"])}<br>Device: {device_pill}</div>')

            # ---------------- main area ----------------
            with gr.Column(scale=1, min_width=600):

                # ===== OVERVIEW =====
                with gr.Column(visible=True) as overview_page:
                    make_html(page_header("Dashboard Overview", "Monitor the supermarket computer-vision pipeline and its latest results.",
                                          [f"Device: {device_pill}", "ByteTrack · YOLOv8"]))
                    init = make_outputs(None, "", None, None, None)
                    with gr.Row():
                        kpi_det = make_html(init[6]); kpi_trk = make_html(init[7]); kpi_frm = make_html(init[8]); kpi_status = make_html(init[9])
                    with gr.Row(equal_height=True):
                        with gr.Column(scale=3):
                            chart_html = make_html(init[11])
                        with gr.Column(scale=2):
                            donut_html = make_html(init[12])
                    with gr.Row(equal_height=True):
                        with gr.Column(scale=2):
                            make_html(pipeline_html())
                        with gr.Column(scale=3):
                            recent_html = make_html(init[13])
                    with gr.Row():
                        quick_studio = gr.Button("Open AI Studio →", variant="primary")
                        quick_missing = gr.Button("Check missing products →", variant="secondary")
                        quick_results = gr.Button("View detection results →", variant="secondary")

                # ===== AI STUDIO =====
                with gr.Column(visible=False) as studio_page:
                    make_html(page_header("AI Studio", "Upload or record a video, choose the settings, and run detection, tracking and classification.", [f"Device: {device_pill}"]))
                    make_html('<div class="sai-banner"><div class="sai-banner-title">Smart retail analysis</div>'
                              '<div class="sai-banner-text">OpenCV · YOLOv8 detection · ByteTrack tracking · Grocery classification</div></div>')
                    with gr.Row(equal_height=False):
                        with gr.Column(scale=1, min_width=330):
                            video_input = gr.Video(label="Video input", sources=["upload", "webcam"], format="mp4", height=340)
                            confidence_input = gr.Slider(CONF_MIN, CONF_MAX, value=DEFAULT_CONF, step=CONF_STEP, label="Detection confidence",
                                                         info="Higher = fewer, more certain detections. Lower = more detections, more false positives.",
                                                         buttons=["reset"], elem_classes=["sai-slider"])
                            interval_input = gr.Radio(FRAME_INTERVAL_CHOICES, value=DEFAULT_INTERVAL, label="Frame processing mode",
                                                      info="Larger intervals are faster but can miss objects between sampled frames and make tracking harder.",
                                                      elem_classes=["sai-radio"])
                            classifier_input = gr.Checkbox(value=True, label="Run grocery classifier on every detected crop (slower)")
                            maxframes_input = gr.Slider(10, 1000, value=DEFAULT_MAX_PROCESSED, step=10, label="Safety limit: maximum frames to analyse",
                                                        info="Stops long videos early so CPU runs stay short.", elem_classes=["sai-slider"])
                            with gr.Row():
                                analyze_button = gr.Button("▶  Analyze video", variant="primary")
                                clear_button = gr.Button("Clear results", variant="secondary")
                        with gr.Column(scale=1, min_width=330):
                            annotated_output = gr.Video(label="Annotated video", height=340, interactive=False)
                            studio_strip = make_html(init[10])
                            summary_output = gr.Markdown("### Analysis summary\n\nResults will appear here after analysis.")
                    studio_table = gr.Dataframe(label="Detection records", value=to_display_df(None), interactive=False, wrap=True, max_height=380)
                    studio_csv = gr.File(label="Download detection records (CSV)")
                    studio_back = gr.Button("← Back to overview", variant="secondary")

                # ===== MISSING PRODUCTS =====
                with gr.Column(visible=False) as missing_page:
                    make_html(page_header("Missing Products", "Compare the products you expect on the shelf with what the model actually detects in the video.",
                                          [f"Device: {device_pill}", "Class-level check"]))
                    make_html(mp_limitations_html())
                    with gr.Row(equal_height=False):
                        with gr.Column(scale=1, min_width=330):
                            mp_ref_input = gr.Image(label="1 · Reference shelf image (optional)", type="numpy", sources=["upload"], height=250)
                            mp_text_input = gr.Textbox(label="2 · Expected products (optional) — one per line", lines=5, placeholder=mp_placeholder(),
                                                       info="Use names from the class list at the bottom of this page" if CLASS_NAMES_AVAILABLE else
                                                            "No class names available: type class_N (see the class list below) or use a reference image")
                        with gr.Column(scale=1, min_width=330):
                            mp_video_input = gr.Video(label="3 · Current shelf video", sources=["upload", "webcam"], format="mp4", height=250)
                            mp_reuse = gr.Checkbox(value=False, label="Re-use the latest analysis (AI Studio or this page) instead of analysing the video again")
                    with gr.Row(equal_height=False):
                        with gr.Column(scale=1, min_width=330):
                            mp_conf = gr.Slider(CONF_MIN, CONF_MAX, value=DEFAULT_CONF, step=CONF_STEP, label="Detection confidence",
                                                info="Applies to the reference image and the video.", buttons=["reset"], elem_classes=["sai-slider"])
                            mp_interval = gr.Radio(FRAME_INTERVAL_CHOICES, value=DEFAULT_INTERVAL, label="Frame processing mode",
                                                   info="Which video frames are analysed.", elem_classes=["sai-radio"])
                        with gr.Column(scale=1, min_width=330):
                            mp_cls_conf = gr.Slider(0.30, 0.95, value=MP_DEFAULT_MIN_CLS_CONF, step=0.05, label="Classifier confidence needed as evidence",
                                                    info="A crop counts as 'seen' only above this classifier confidence.", buttons=["reset"], elem_classes=["sai-slider"])
                            mp_min_frames = gr.Slider(1, 10, value=MP_DEFAULT_MIN_FRAMES, step=1, label="Frames needed to confirm a product",
                                                      info="Analysed frames in which it must be seen to count as Observed.", elem_classes=["sai-slider"])
                            mp_maxframes = gr.Slider(10, 1000, value=DEFAULT_MAX_PROCESSED, step=10, label="Safety limit: maximum frames to analyse",
                                                     elem_classes=["sai-slider"])
                    with gr.Row():
                        mp_run = gr.Button("▶  Compare expected vs. detected", variant="primary")
                        mp_clear = gr.Button("Clear results", variant="secondary")
                    mp_summary = make_html(mp_summary_html(None))
                    mp_results = make_html(mp_results_html(None))
                    mp_csv = gr.File(label="Download comparison (CSV)")
                    mp_gallery = gr.Gallery(label="Evidence crops (cut from the model's own detections)", columns=4, height=280, object_fit="contain", interactive=False)
                    with gr.Row(equal_height=False):
                        mp_ref_preview = gr.Image(label="Reference image: detections + classifier labels", interactive=False, height=300)
                        mp_video_out = gr.Video(label="Annotated shelf video", interactive=False, height=300)
                    with gr.Accordion("Full comparison table", open=False):
                        mp_table = gr.Dataframe(value=_mp_display(None), interactive=False, wrap=True, max_height=380)
                    make_html(mp_class_list_html())

                # ===== RESULTS =====
                with gr.Column(visible=False) as results_page:
                    make_html(page_header("Detection Results", "Every record produced by the latest analysis.", []))
                    results_note = make_html(init[14])
                    results_table = gr.Dataframe(label="Detection records", value=to_display_df(None), interactive=False, wrap=True, max_height=520)
                    results_csv = gr.File(label="Download detection records (CSV)")
                    with gr.Row():
                        results_back = gr.Button("← Back to AI Studio", variant="primary")
                        results_overview = gr.Button("Dashboard overview", variant="secondary")

                # ===== ABOUT =====
                with gr.Column(visible=False) as about_page:
                    make_html(page_header("About the Pipeline", "How this prototype turns a video into detection records.", []))
                    with gr.Row(equal_height=True):
                        with gr.Column(scale=3):
                            make_html(about_html())
                        with gr.Column(scale=2):
                            make_html(limits_html())

        # ---------------- navigation (page visibility + highlighted nav button) ----------------
        pages = [overview_page, studio_page, missing_page, results_page, about_page]       # order = PAGE_* constants
        assert len(pages) == len(NAV_ITEMS) == len(nav_btns)
        def navigate(idx):
            return [gr.update(visible=(i == idx)) for i in range(len(pages))] + [gr.update(variant=("primary" if i == idx else "secondary")) for i in range(len(pages))]
        nav_outputs = pages + nav_btns
        for btn, (_, idx) in zip(nav_btns, NAV_ITEMS):
            btn.click(lambda i=idx: navigate(i), outputs=nav_outputs, show_progress="hidden")
        quick_studio.click(lambda: navigate(PAGE_STUDIO), outputs=nav_outputs, show_progress="hidden")
        quick_missing.click(lambda: navigate(PAGE_MISSING), outputs=nav_outputs, show_progress="hidden")
        quick_results.click(lambda: navigate(PAGE_RESULTS), outputs=nav_outputs, show_progress="hidden")
        studio_back.click(lambda: navigate(PAGE_OVERVIEW), outputs=nav_outputs, show_progress="hidden")
        results_back.click(lambda: navigate(PAGE_STUDIO), outputs=nav_outputs, show_progress="hidden")
        results_overview.click(lambda: navigate(PAGE_OVERVIEW), outputs=nav_outputs, show_progress="hidden")

        # ---------------- analysis ----------------
        analysis_outputs = [annotated_output, summary_output, studio_table, studio_csv, results_table, results_csv,
                            kpi_det, kpi_trk, kpi_frm, kpi_status, studio_strip, chart_html, donut_html, recent_html, results_note]
        assert len(analysis_outputs) == N_OUTPUTS
        analysis_outputs = analysis_outputs + [analysis_state]
        analyze_button.click(run_analysis,
                             inputs=[video_input, confidence_input, interval_input, classifier_input, maxframes_input, analysis_state],
                             outputs=analysis_outputs)
        clear_button.click(clear_results, outputs=analysis_outputs)

        # ---------------- missing products ----------------
        mp_outputs_list = [mp_summary, mp_results, mp_table, mp_csv, mp_gallery, mp_ref_preview, mp_video_out, analysis_state]
        assert len(mp_outputs_list) == MP_N_OUTPUTS
        mp_run.click(run_missing_products,
                     inputs=[mp_ref_input, mp_text_input, mp_video_input, mp_reuse, mp_conf, mp_interval, mp_maxframes, mp_cls_conf, mp_min_frames, analysis_state],
                     outputs=mp_outputs_list)
        mp_clear.click(clear_missing, outputs=mp_outputs_list)

    demo.queue(default_concurrency_limit=1)     # one analysis at a time
    return demo, launch_kw

demo, LAUNCH_STYLE_KW = build_dashboard()
print("Dashboard built. Next cell launches it.")

## 10. Quick self-test (optional)
Runs the real pipeline on a tiny generated video to confirm frame-interval handling, output video length and CSV export. Set `RUN_SMOKE_TEST = False` to skip.

In [ ]:
# ---- Optional wiring test (takes seconds): runs the REAL pipeline on a small generated video ----
# It checks: frame-interval handling, frame numbering, output video length, CSV creation and the empty-result path.
# The generated video is just moving rectangles, so the detector may find nothing - that is fine and tests the "no detections" path.
RUN_SMOKE_TEST = True

def make_test_video(path, n_frames=45, w=320, h=240, fps=15):
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for i in range(n_frames):
        img = np.full((h, w, 3), 235, np.uint8)
        cv2.rectangle(img, (20 + i * 3, 60), (80 + i * 3, 160), (40, 120, 200), -1)
        wr.write(img)
    wr.release()
    return path, n_frames

if RUN_SMOKE_TEST:
    test_path, n = make_test_video(os.path.join(OUTPUT_DIR, "smoke_test_input.mp4"))
    for interval in (1, 3, 5):
        out_video, summary, df_t, csv_t, st = analyze_video(test_path, 0.25, interval, use_classifier=True, max_processed_frames=1000)
        expected = math.ceil(n / interval)
        assert st["processed_frames"] == expected, (interval, st["processed_frames"], expected)
        assert os.path.isfile(csv_t) and os.path.isfile(out_video)
        cap = cv2.VideoCapture(out_video); written = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        assert abs(written - n) <= 1, f"output video has {written} frames, input had {n}"
        print(f"interval {interval}: analysed {st['processed_frames']}/{n} frames, output video {written} frames, "
              f"{st['detections']} detections, {st['unique_tracks']} tracks, CSV columns: {len(df_t.columns)}")
    # the confidence slider must reach ByteTrack (bundled default would silently ignore everything below 0.25)
    _p, _note = make_tracker_yaml(0.15)
    _cfg = yaml.safe_load(open(_p))
    assert abs(_cfg["track_high_thresh"] - 0.15) < 1e-9 and abs(_cfg["new_track_thresh"] - 0.15) < 1e-9, _cfg
    print("tracker thresholds follow the confidence slider:", _note)
    print("\nSmoke test passed. (Detections on the generated video are not meaningful - use a real shelf video in the dashboard.)")

## 10b. Missing Products logic self-test (optional)
Checks the decision rules with **synthetic records** (clearly fake test fixtures, never shown on the dashboard) so you can see the rules behave as documented without needing a video.

In [ ]:
# ---- Missing Products: decision-rule self-test on SYNTHETIC records (test fixtures only - NOT model output) ----
RUN_MP_SELFTEST = True

if RUN_MP_SELFTEST:
    _names = ["t_zero", "t_one", "t_two", "t_three", "t_four", "t_five"]       # fake class vocabulary, test only
    def _rec(frame, cid, conf, tid):
        return dict(frame_number=frame, timestamp_s=frame / 10, track_id=tid, detector_class="object", detector_confidence=0.9,
                    x1=0, y1=0, x2=10, y2=10, classifier_class_id=cid, classifier_label=_names[cid], classifier_confidence=conf)
    _rows = ([_rec(f, 0, 0.95, 1) for f in (0, 3, 6)]            # class 0: strong in 3 frames          -> Observed
             + [_rec(3, 1, 0.90, 2)]                              # class 1: strong in only 1 frame       -> Inconclusive
             + [_rec(f, 2, 0.30, 3) for f in (0, 3, 6)]           # class 2: seen only at low confidence  -> Inconclusive
             + [_rec(f, 4, 0.90, 4) for f in range(0, 30, 3)])    # class 4: filler so the video has enough reliable detections
    _df = pd.DataFrame(_rows)
    _stats = dict(processed_frames=10, fps=10.0, confidence=0.4, frame_interval=3)
    _exp = [dict(product=_names[i], source="reference image", class_id=i, reference_instances=2, ref_best_conf=0.9, reference_ok=True, note="") for i in (0, 1, 2, 3)]
    _exp.append(dict(product="unmapped thing", source="typed list", class_id=None, reference_instances=None, ref_best_conf=None, reference_ok=True, note="no classifier class has this name"))
    _exp.append(dict(product=_names[5], source="reference image", class_id=5, reference_instances=0, ref_best_conf=0.4, reference_ok=False, note="weak reference"))
    _res = compare_expected_vs_observed(_exp, _df, _stats, _names, min_cls_conf=0.6, min_frames=2).set_index("product")["status"].to_dict()
    assert _res["t_zero"] == MP_STATUS_OBSERVED, _res
    assert _res["t_one"] == MP_STATUS_INCONCLUSIVE, _res          # too few frames
    assert _res["t_two"] == MP_STATUS_INCONCLUSIVE, _res          # low confidence only
    assert _res["t_three"] == MP_STATUS_MISSING, _res             # never detected, plenty of evidence
    assert _res["unmapped thing"] == MP_STATUS_INCONCLUSIVE, _res
    assert _res["t_five"] == MP_STATUS_INCONCLUSIVE, _res         # weak reference
    # absence must NOT be called 'missing' when the video is too thin or the classifier is unsure
    _thin = compare_expected_vs_observed(_exp[3:4], _df.head(2), dict(_stats, processed_frames=2), _names, 0.6, 2)
    assert _thin.loc[0, "status"] == MP_STATUS_INCONCLUSIVE, _thin
    _unsure = compare_expected_vs_observed(_exp[3:4], _df.assign(classifier_confidence=0.2), _stats, _names, 0.6, 2)
    assert _unsure.loc[0, "status"] == MP_STATUS_INCONCLUSIVE, _unsure
    # typed-name parsing / matching
    assert parse_expected_text("- a\n2) b\n\nA\n") == ["a", "b"]
    assert parse_expected_text("x, y, z") == ["x", "y", "z"]
    assert map_name_to_class("class_3", _names, False)[0] == 3
    assert map_name_to_class("t_three", _names, True)[0] == 3 and map_name_to_class("nothing here", _names, True)[0] is None
    assert map_name_to_class("t_three", _names, False)[0] is None                     # no names file -> cannot match by name
    print("Missing Products logic self-test passed (synthetic fixtures only).")
    # diagnostics: the hints must come from the real numbers (synthetic fixtures again)
    _d0 = build_mp_diagnostics("t", True, pd.DataFrame(columns=REF_DET_COLUMNS), _df, _stats, None, 0.4, 0.6, 2)
    assert _d0["reference"]["detections"] == 0 and any("no objects in the reference" in h for h in _d0["hints"]), _d0
    _weak = _df.assign(classifier_confidence=0.45)
    _d1 = build_mp_diagnostics("t", False, None, _weak, _stats, None, 0.4, 0.6, 2)
    assert _d1["video"]["kept_at_classifier_threshold"] == 0 and any("0.45" in h for h in _d1["hints"]), _d1
    assert "Diagnostics" in mp_diagnostics_html(_d1)
    print("Missing Products diagnostics self-test passed (synthetic fixtures only).")
    # observed-products algorithm (works with NO baseline): per-class aggregation of the video's own records
    _obs = summarize_observed_products(_df, _stats, _names, min_cls_conf=0.6, min_frames=2).set_index("product")
    assert _obs.loc["t_zero", "status"] == OBS_CONFIDENT and _obs.loc["t_zero", "frames_confident"] == 3 and _obs.loc["t_zero", "video_tracks"] == 1, _obs
    assert _obs.loc["t_one", "status"] == OBS_WEAK and _obs.loc["t_two", "status"] == OBS_WEAK, _obs
    assert "t_three" not in _obs.index, "a class that was never detected must not be listed"
    assert len(summarize_observed_products(_df.head(0), _stats, _names, 0.6, 2)) == 0
    print("Missing Products observed-products self-test passed (synthetic fixtures only).")


## 11. Launch
Run this cell last. It stays running (`debug=True`) so errors appear here. **Open the printed public URL in a new tab** for the full-size dashboard. To restart, stop the cell and run it again — `gr.close_all()` clears any old server first.

In [ ]:
# ---- Launch the dashboard ----
# * gr.close_all() stops any server left over from an earlier run (stale servers are a classic cause of
#   "Unexpected token '<'" errors, because the browser then receives an HTML error page instead of JSON).
# * debug=True keeps this cell running and prints errors/tracebacks here. Stop the cell (square button) to shut the server down.
# * In Colab, the inline preview is small: open the public URL printed below in a NEW TAB for the full dashboard.

USE_SHARE_LINK = IN_COLAB      # Colab needs a share link (or its proxy) to reach the app from your browser; set False when running locally

gr.close_all()
demo.launch(
    share=USE_SHARE_LINK,
    debug=True,
    show_error=True,
    allowed_paths=[OUTPUT_DIR],     # lets Gradio serve the annotated videos and CSVs we create
    **LAUNCH_STYLE_KW,              # theme / css / head (Gradio 6 takes them in launch())
)

## Troubleshooting
| Symptom | Cause / fix |
|---|---|
| `Could not parse server response: SyntaxError: Unexpected token '<'` | The browser received an HTML error page instead of JSON: usually a stale/crashed server or an expired/blocked share link. Stop the launch cell, re-run it (it closes old servers), and open the new URL. Check the cell output for a Python traceback. |
| Text hard to read | Make sure step 9 ran after step 2 (restart + run all). Theme and CSS are applied at launch. |
| Annotated video does not play | `ffmpeg` re-encode failed; the summary names the reason. The CSV and table are unaffected. |
| Analysis is slow | Use a GPU runtime, a larger frame interval, turn the classifier off, or lower the frame limit. |
| Classifier labels are `class_N` | Expected until you supply `class_names.json` (step 6). |
| Missing Products shows only *Inconclusive* | By design: the video gave too little evidence (few frames/detections, or the classifier is rarely confident), the typed name matches no classifier class, or the reference crops were weak. Read the *Evidence* column; try a lower frame interval, a lower classifier confidence, or supply `class_names.json`. |
| Typed product names are not matched | No class-name mapping exists. Type `class_N` (see the class list at the bottom of the page) or use a reference image. |
